# Factor-DDPM Market Simulator Colab V8


## V8 hardening contract

This notebook is the methodological break from v7. It keeps the v7 factor-DDPM spine, but removes the two behaviors that made the old stress book too easy to pass:

- no deterministic full-window severe clamp;
- no default q0.001/q0.999 tail winsorization;
- no checkpoint selection on noise MSE alone;
- no promotion unless DDPM beats Gaussian, same-stack Gaussian, t-copula, filtered historical simulation, and bootstrap baselines on tail/correlation gates;
- no overlapping-window MMD headline;
- factor scenario bank export is first-class, so the endpoint can serve real sampled factors without a GPU.


## 0. Colab Runtime

Recommended runtime:
- GPU: A100 if available; T4/L4 works for smaller universes.
- Runtime type: Python 3 with GPU.

Default settings are conservative enough for Colab. Increase `N_ASSETS`, `N_EPOCHS`, and `N_SCENARIOS` only after the first end-to-end run succeeds.
        


In [ ]:
#@title Install dependencies
import sys, subprocess, os, textwrap, json, math, random, time

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("numpy", "pandas", "scikit-learn", "matplotlib", "tqdm", "requests", "hmmlearn", "yfinance")


In [ ]:
#@title Imports and device
import os, json, math, time, random, warnings, re, gc
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Tuple
import getpass
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from hmmlearn.hmm import GaussianHMM
    HMM_AVAILABLE = True
except Exception as exc:
    HMM_AVAILABLE = False
    GaussianHMM = None
    print("hmmlearn unavailable; regime detector will fall back to KMeans:", exc)

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except Exception as exc:
    YFINANCE_AVAILABLE = False
    yf = None
    print("yfinance unavailable; macro/VIX conditioning will use internal market features only:", exc)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/blsprime_ddpm_market_sim")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for path in [DATA_DIR, ARTIFACT_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Put your CSV in:", DATA_DIR)



## 1. Configuration

Input options:
- `DATA_SOURCE = "fmp"` downloads historical EOD prices from Financial Modeling Prep into Drive.
- `DATA_SOURCE = "csv"` loads a CSV from Drive.
- `DATA_SOURCE = "demo"` creates a synthetic demo dataset.

For CSV:
- first column is date or index;
- remaining columns are either adjusted prices or returns.

If `DATA_MODE = "prices"`, the notebook computes log returns.
If `DATA_MODE = "returns"`, the notebook uses the numeric columns directly.
        


In [ ]:
#@title User configuration
DATA_SOURCE = "fmp"  #@param ["fmp", "csv", "demo"]
CSV_FILENAME = "market_prices_or_returns.csv"  #@param {type:"string"}
CSV_PATH = DATA_DIR / CSV_FILENAME
DATA_MODE = "prices"  #@param ["prices", "returns"]

# FMP source. The next cell replaces this with the full S&P 500 universe.
FMP_SYMBOLS = "AUTO_SP500"  #@param {type:"string"}
FMP_START_DATE = "2010-01-01"  #@param {type:"string"}
FMP_END_DATE = ""  # blank = today
FMP_SLEEP_SECONDS = 0.12

# Universe / sample construction
N_ASSETS = 500         #@param {type:"integer"}
WINDOW_SIZE = 30       #@param {type:"integer"}
N_REGIMES = 4          #@param {type:"integer"}
VALIDATION_MODE = "walk_forward_crisis"  #@param ["walk_forward_crisis", "temporal_80_20"]
TRAIN_FRACTION = 0.80  # used only when VALIDATION_MODE="temporal_80_20"
WALK_FORWARD_TRAIN_END = "2019-12-31"  # train before COVID/inflation stress, validate after
MIN_HISTORY_FRACTION = 0.70
REGIME_METHOD = "hmm"  #@param ["hmm", "kmeans"]
USE_MACRO_CONDITIONING = True

# Professional factor engine. The DDPM learns market/sector/PCA factors, then reconstructs assets.
MODEL_FAMILY = "factor_ddpm_v8_tail_hardened_factor_bank"
FACTOR_PCA_COMPONENTS = 32
FACTOR_RIDGE_ALPHA = 1.0e-3
RESIDUAL_COV_SHRINKAGE = 0.25
RESIDUAL_BOOTSTRAP_SCALE = 1.00
STRESS_SCENARIO_MODE = True
STRESS_STRATIFIED_SAMPLING = True
STRESS_MIX_WEIGHTS = "0.63,0.22,0.10,0.05"
STRESS_MIX_MULTIPLIERS = "1.0,1.45,2.4,6.0"
DOWNSIDE_ASYMMETRY = 1.35
FACTOR_CALIBRATION_ALPHA = 0.45
FACTOR_CALIBRATION_SHRINKAGE = 0.20
FACTOR_CORR_LOSS_WEIGHT = 0.20
ASSET_CORR_LOSS_WEIGHT = 0.35
PORTFOLIO_TAIL_LOSS_WEIGHT = 0.75
CORR_PROJECTION_DIM = 128

# DDPM
N_TIMESTEPS = 600      #@param {type:"integer"}
BETA_SCHEDULE = "cosine"  #@param ["cosine", "linear"]
P_UNCOND = 0.12
MIN_SNR_GAMMA = 5.0
TAIL_ALPHA = 1.35
TAIL_LAMBDA = 1.00
TAIL_WEIGHT_CLIP = 18.0
CORR_LOSS_WEIGHT = 0.18

# Model. A100 profile. If Colab gives T4/L4: D_MODEL=160, N_LAYERS=6, BATCH_SIZE=96.
D_MODEL = 320
N_HEADS = 8
N_LAYERS = 8
D_FF = 1280
DROPOUT = 0.08

# Training
N_EPOCHS = 180
BATCH_SIZE = 96
LR = 1.0e-4
MIN_LR_RATIO = 0.08
WARMUP_EPOCHS = 8
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
EMA_DECAY = 0.9997
NUM_WORKERS = 0
USE_REGIME_BALANCED_SAMPLER = True
REGIME_SAMPLER_POWER = 0.70
REGIME_LOSS_POWER = 0.50
TARGET_REGIME_LOSS_BOOST = 1.75
EARLY_STOP_PATIENCE = 45
EARLY_STOP_MIN_DELTA = 1e-4
MIN_EPOCHS_BEFORE_STOP = 90

# Sampling
N_SCENARIOS = 5000
SAMPLE_BATCH_SIZE = 512
DDIM_STEPS = 120
DDIM_ETA = 0.15
GUIDANCE_SCALE = 0.6
AUTO_GUIDANCE_SWEEP = True
GUIDANCE_SWEEP = "0.0,0.3,0.6,1.0,1.4,2.0"
GUIDANCE_SWEEP_SCENARIOS = 512
APPLY_CHOLESKY_CALIBRATION = True
CHOLESKY_CALIBRATION_ALPHA = 0.40
CHOLESKY_SHRINKAGE = 0.10
TARGET_REGIME = 2
AUTO_TARGET_CRISIS_REGIME = True

# Evaluation
BASELINE_SCENARIOS = 2500
MMD_PROJECTION_DIM = 128
MMD_MAX_WINDOWS = 1500
MMD_STABILITY_SEEDS = "11,23,37,53,71"
EVAL_SYNTHETIC_SUBSAMPLE_SEEDS = "101,203,307,409,503"
PRIMARY_EVAL_SUBSAMPLE_SEED = 101
SKIPPED_BATCH_RATE_MAX = 0.001
CORR_MAE_NEAR_GAUSSIAN_TOL = 0.005
MMD_RATIO_MAX_RESEARCH = 1.05

# Endpoint/research deployment contract. Endpoint stays gated until strict production metrics pass.
ENDPOINT_DEFAULT_SCENARIOS = 5000
ENDPOINT_MIN_SCENARIOS = 2000
ENDPOINT_MIN_STRESS_SCENARIOS = 5000
ENDPOINT_STRESS_QUANTILE = 0.01
ENDPOINT_SMALL_REQUEST_POLICY = "reject_or_aggregate"

# V8 methodology hardening.
RETURN_CLIP_MODE = "bad_print_only"  #@param ["none", "bad_print_only", "wide_quantile"]
RETURN_CLIP_LOWER_Q = 0.0001
RETURN_CLIP_UPPER_Q = 0.9999
BAD_PRINT_ABS_RETURN_LIMIT = 0.80
EVAL_WINDOW_STRIDE = WINDOW_SIZE
CHECKPOINT_SELECTION_METRIC = "valid_tail_composite"
VALID_SELECTION_MAX_BATCHES = 30
STRESS_SHOCK_MODE = "sparse_distribution_shift"  # no full-window deterministic floor
SEVERE_SHOCK_MIN_DAYS = 2
SEVERE_SHOCK_MAX_DAYS = 5
SEVERE_MARKET_LOCATION_SHIFT_Q = 0.01
SEVERE_MARKET_SCALE_BOOST = 1.35
BASELINE_T_COPULA_DF = 5
FHS_EWMA_LAMBDA = 0.94
PORTFOLIO_TEST_COUNT = 32
PORTFOLIO_CONCENTRATION_LEVELS = "0.25,0.40,0.60"
EXPORT_FACTOR_SCENARIO_BANK = True
FACTOR_BANK_DTYPE = "float16"
FACTOR_BANK_REGIME_NAME = "crisis"
SURVIVORSHIP_DISCLOSURE = "current_sp500_constituents_only_unless_pit_universe_supplied"

CONFIG = dict(
    seed=SEED, data_source=DATA_SOURCE, data_mode=DATA_MODE, fmp_symbols=FMP_SYMBOLS,
    fmp_start_date=FMP_START_DATE, fmp_end_date=FMP_END_DATE, csv_filename=CSV_FILENAME,
    n_assets=N_ASSETS, window_size=WINDOW_SIZE, min_history_fraction=MIN_HISTORY_FRACTION,
    validation_mode=VALIDATION_MODE, train_fraction=TRAIN_FRACTION,
    walk_forward_train_end=WALK_FORWARD_TRAIN_END,
    regime_method=REGIME_METHOD, use_macro_conditioning=USE_MACRO_CONDITIONING,
    n_regimes=N_REGIMES, n_timesteps=N_TIMESTEPS, beta_schedule=BETA_SCHEDULE,
    p_uncond=P_UNCOND, min_snr_gamma=MIN_SNR_GAMMA,
    tail_alpha=TAIL_ALPHA, tail_lambda=TAIL_LAMBDA, tail_weight_clip=TAIL_WEIGHT_CLIP,
    corr_loss_weight=CORR_LOSS_WEIGHT, factor_corr_loss_weight=FACTOR_CORR_LOSS_WEIGHT,
    asset_corr_loss_weight=ASSET_CORR_LOSS_WEIGHT, portfolio_tail_loss_weight=PORTFOLIO_TAIL_LOSS_WEIGHT,
    model_family=MODEL_FAMILY, factor_pca_components=FACTOR_PCA_COMPONENTS,
    factor_ridge_alpha=FACTOR_RIDGE_ALPHA, residual_cov_shrinkage=RESIDUAL_COV_SHRINKAGE,
    residual_bootstrap_scale=RESIDUAL_BOOTSTRAP_SCALE, stress_scenario_mode=STRESS_SCENARIO_MODE,
    stress_stratified_sampling=STRESS_STRATIFIED_SAMPLING,
    stress_mix_weights=STRESS_MIX_WEIGHTS, stress_mix_multipliers=STRESS_MIX_MULTIPLIERS,
    downside_asymmetry=DOWNSIDE_ASYMMETRY, factor_calibration_alpha=FACTOR_CALIBRATION_ALPHA,
    factor_calibration_shrinkage=FACTOR_CALIBRATION_SHRINKAGE,
    corr_projection_dim=CORR_PROJECTION_DIM,
    d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT, n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR, min_lr_ratio=MIN_LR_RATIO,
    warmup_epochs=WARMUP_EPOCHS, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
    ema_decay=EMA_DECAY, use_regime_balanced_sampler=USE_REGIME_BALANCED_SAMPLER,
    regime_sampler_power=REGIME_SAMPLER_POWER, regime_loss_power=REGIME_LOSS_POWER,
    target_regime_loss_boost=TARGET_REGIME_LOSS_BOOST,
    n_scenarios=N_SCENARIOS, sample_batch_size=SAMPLE_BATCH_SIZE,
    ddim_steps=DDIM_STEPS, ddim_eta=DDIM_ETA,
    guidance_scale=GUIDANCE_SCALE, auto_guidance_sweep=AUTO_GUIDANCE_SWEEP,
    guidance_sweep=GUIDANCE_SWEEP, guidance_sweep_scenarios=GUIDANCE_SWEEP_SCENARIOS,
    apply_cholesky_calibration=APPLY_CHOLESKY_CALIBRATION,
    cholesky_calibration_alpha=CHOLESKY_CALIBRATION_ALPHA,
    cholesky_shrinkage=CHOLESKY_SHRINKAGE,
    target_regime=TARGET_REGIME, baseline_scenarios=BASELINE_SCENARIOS,
    mmd_projection_dim=MMD_PROJECTION_DIM, mmd_max_windows=MMD_MAX_WINDOWS,
    mmd_stability_seeds=MMD_STABILITY_SEEDS,
    eval_synthetic_subsample_seeds=EVAL_SYNTHETIC_SUBSAMPLE_SEEDS,
    primary_eval_subsample_seed=PRIMARY_EVAL_SUBSAMPLE_SEED,
    skipped_batch_rate_max=SKIPPED_BATCH_RATE_MAX,
    corr_mae_near_gaussian_tol=CORR_MAE_NEAR_GAUSSIAN_TOL,
    mmd_ratio_max_research=MMD_RATIO_MAX_RESEARCH,
    endpoint_default_scenarios=ENDPOINT_DEFAULT_SCENARIOS,
    endpoint_min_scenarios=ENDPOINT_MIN_SCENARIOS,
    endpoint_min_stress_scenarios=ENDPOINT_MIN_STRESS_SCENARIOS,
    endpoint_stress_quantile=ENDPOINT_STRESS_QUANTILE,
    endpoint_small_request_policy=ENDPOINT_SMALL_REQUEST_POLICY,
    return_clip_mode=RETURN_CLIP_MODE,
    return_clip_lower_q=RETURN_CLIP_LOWER_Q,
    return_clip_upper_q=RETURN_CLIP_UPPER_Q,
    bad_print_abs_return_limit=BAD_PRINT_ABS_RETURN_LIMIT,
    eval_window_stride=EVAL_WINDOW_STRIDE,
    checkpoint_selection_metric=CHECKPOINT_SELECTION_METRIC,
    valid_selection_max_batches=VALID_SELECTION_MAX_BATCHES,
    stress_shock_mode=STRESS_SHOCK_MODE,
    severe_shock_min_days=SEVERE_SHOCK_MIN_DAYS,
    severe_shock_max_days=SEVERE_SHOCK_MAX_DAYS,
    severe_market_location_shift_q=SEVERE_MARKET_LOCATION_SHIFT_Q,
    severe_market_scale_boost=SEVERE_MARKET_SCALE_BOOST,
    baseline_t_copula_df=BASELINE_T_COPULA_DF,
    fhs_ewma_lambda=FHS_EWMA_LAMBDA,
    portfolio_test_count=PORTFOLIO_TEST_COUNT,
    portfolio_concentration_levels=PORTFOLIO_CONCENTRATION_LEVELS,
    export_factor_scenario_bank=EXPORT_FACTOR_SCENARIO_BANK,
    factor_bank_dtype=FACTOR_BANK_DTYPE,
    factor_bank_regime_name=FACTOR_BANK_REGIME_NAME,
    survivorship_disclosure=SURVIVORSHIP_DISCLOSURE,
)
print(json.dumps(CONFIG, indent=2))


In [ ]:
#@title Large universe: FMP -> DataHub S&P 500 -> Wikipedia -> static fallback
import io
import re

# Practical defaults:
# - T4/L4: MAX_FMP_SYMBOLS=250-350, BATCH_SIZE=96-128, D_MODEL=160
# - A100: MAX_FMP_SYMBOLS=500, BATCH_SIZE=192, D_MODEL=192
UNIVERSE_SOURCE = "sp500"
MAX_FMP_SYMBOLS = 500
DATAHUB_SP500_URL = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
STATIC_SP500_SYMBOLS = """MMM AOS ABT ABBV ACN ADBE AMD AES AFL A APD ABNB AKAM ALB ARE ALGN ALLE LNT ALL GOOGL GOOG MO AMZN AMCR AEE AEP AXP AIG AMT AWK AMP AME AMGN APH ADI AON APA APO AAPL AMAT APP APTV ACGL ADM ARES ANET AJG AIZ T ATO ADSK ADP AZO AVB AVY AXON BKR BALL BAC BAX BDX BRK-B BBY TECH BIIB BLK BX XYZ BNY BA BKNG BSX BMY AVGO BR BRO BF-B BLDR BG BXP CHRW CDNS CPT COF CAH CCL CARR CVNA CASY CAT CBOE CBRE CDW COR CNC CNP CF CRL SCHW CHTR CVX CMG CB CHD CIEN CI CINF CTAS CSCO C CFG CLX CME CMS KO CTSH COHR COIN CL CMCSA FIX COP ED STZ CEG COO CPRT GLW CPAY CTVA CSGP COST CRH CRWD CCI CSX CMI CVS DHR DRI DDOG DVA DECK DE DELL DAL DVN DXCM FANG DLR DG DLTR D DPZ DASH DOV DOW DHI DTE DUK DD ETN EBAY ECHO ECL EIX EW EA ELV EME EMR ETR EOG EQT EFX EQIX EQR ERIE ESS EL EG EVRG ES EXC EXE EXPE EXPD EXR XOM FFIV FDS FICO FAST FRT FDX FDXF FIS FITB FSLR FE FISV FLEX F FTNT FTV FOXA FOX BEN FCX GRMN IT GE GEHC GEV GEN GNRC GD GIS GM GPC GILD GPN GL GDDY GS HAL HIG HAS HCA DOC HSIC HSY HPE HLT HD HONA HON HRL HST HWM HPQ HUBB HUM HBAN HII IBM IEX IDXX ITW INCY IR PODD INTC IBKR ICE IFF IP INTU ISRG IVZ INVH IQV IRM JBHT JBL JKHY J JNJ JCI JPM KVUE KDP KEY KEYS KMB KIM KMI KKR KLAC KHC KR LHX LH LRCX LVS LDOS LEN LII LLY LIN LYV LMT L LOW LULU LITE LYB MTB MPC MAR MRSH MLM MRVL MAS MA MKC MCD MCK MDT MRK META MET MTD MGM MCHP MU MSFT MAA MRNA TAP MDLZ MPWR MNST MCO MS MOS MSI MSCI NDAQ NTAP NFLX NEM NWSA NWS NEE NKE NI NDSN NSC NTRS NOC NCLH NRG NUE NVDA NVR NXPI ORLY OXY ODFL OMC ON OKE ORCL OTIS PCAR PKG PLTR PANW PSKY PH PAYX PYPL PNR PEP PFE PCG PM PSX PNW PNC PPG PPL PFG PG PGR PLD PRU PEG PTC PSA PHM PWR QCOM DGX Q RL RJF RTX O REG REGN RF RSG RMD RVTY HOOD ROK ROL ROP ROST RCL SPGI CRM SNDK SBAC SLB STX SRE NOW SHW SPG SWKS SJM SW SNA SOLV SO LUV SWK SBUX STT STLD STE SYK SMCI SYF SNPS SYY TMUS TROW TTWO TPR TRGP TGT TEL TDY TER TSLA TXN TPL TXT TMO TJX TKO TTD TSCO TT TDG TRV TRMB TFC TYL TSN USB UBER UDR ULTA UNP UAL UPS URI UNH UHS VLO VEEV VTR VLTO VRSN VRSK VZ VRTX VRT VTRS VICI V VST VMC WRB GWW WAB WMT DIS WBD WM WAT WEC WFC WELL WST WDC WY WSM WMB WTW WDAY WYNN XEL XYL YUM ZBRA ZBH ZTS""".split()

def sanitize_error_message(exc, api_key=None):
    text = str(exc)
    if api_key:
        text = text.replace(api_key, "***")
    text = re.sub(r"apikey=[^&\s]+", "apikey=***", text)
    return text[:220]

def get_fmp_api_key():
    key = os.environ.get("FMP_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = (userdata.get("FMP_API_KEY") or "").strip()
        if key:
            os.environ["FMP_API_KEY"] = key
            return key
    except Exception:
        pass
    key = getpass.getpass("Paste FMP API key (input hidden): ").strip()
    os.environ["FMP_API_KEY"] = key
    return key

def clean_symbols(symbols):
    out = []
    for symbol in symbols:
        symbol = str(symbol).upper().strip().replace(".", "-")
        if symbol and re.match(r"^[A-Z0-9\-]+$", symbol):
            out.append(symbol)
    return list(dict.fromkeys(out))

def try_fmp_sp500_constituents():
    api_key = get_fmp_api_key()
    endpoints = [
        ("stable", "https://financialmodelingprep.com/stable/sp500-constituent"),
        ("legacy", "https://financialmodelingprep.com/api/v3/sp500_constituent"),
    ]
    for label, url in endpoints:
        try:
            response = requests.get(url, params={"apikey": api_key}, timeout=30)
            if response.status_code in (401, 402, 403):
                print(f"FMP {label} constituents blocked by plan/key: {response.status_code}")
                continue
            response.raise_for_status()
            payload = response.json()
            if isinstance(payload, list) and payload:
                frame = pd.DataFrame(payload)
                symbol_col = "symbol" if "symbol" in frame.columns else frame.columns[0]
                symbols = clean_symbols(frame[symbol_col].tolist())
                if symbols:
                    print(f"Universe from FMP {label}: {len(symbols)} symbols")
                    return symbols, f"fmp_{label}"
        except Exception as exc:
            print(f"FMP {label} failed:", sanitize_error_message(exc, api_key))
    return [], None

def try_datahub_sp500_constituents():
    try:
        response = requests.get(
            DATAHUB_SP500_URL,
            headers={"User-Agent": "Mozilla/5.0 ddpm-market-simulator"},
            timeout=30,
        )
        response.raise_for_status()
        frame = pd.read_csv(io.StringIO(response.text))
        symbol_col = "Symbol" if "Symbol" in frame.columns else frame.columns[0]
        symbols = clean_symbols(frame[symbol_col].tolist())
        if symbols:
            print(f"Universe from DataHub S&P 500 CSV: {len(symbols)} symbols")
            return symbols, "datahub_sp500"
    except Exception as exc:
        print("DataHub fallback failed:", sanitize_error_message(exc))
    return [], None

def try_wikipedia_sp500_constituents():
    try:
        response = requests.get(
            "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
            headers={"User-Agent": "Mozilla/5.0 ddpm-market-simulator"},
            timeout=30,
        )
        response.raise_for_status()
        tables = pd.read_html(io.StringIO(response.text))
        frame = tables[0]
        symbol_col = "Symbol" if "Symbol" in frame.columns else frame.columns[0]
        symbols = clean_symbols(frame[symbol_col].tolist())
        if symbols:
            print(f"Universe from Wikipedia S&P 500: {len(symbols)} symbols")
            return symbols, "wikipedia_sp500"
    except Exception as exc:
        print("Wikipedia fallback failed:", sanitize_error_message(exc))
    return [], None

def fallback_static_sp500_universe():
    symbols = clean_symbols(STATIC_SP500_SYMBOLS)
    print(f"Universe from embedded static S&P 500 fallback: {len(symbols)} symbols")
    return symbols, "embedded_sp500_snapshot"

symbols, resolved_source = [], None
if UNIVERSE_SOURCE == "sp500":
    symbols, resolved_source = try_fmp_sp500_constituents()
    if not symbols:
        symbols, resolved_source = try_datahub_sp500_constituents()
    if not symbols:
        symbols, resolved_source = try_wikipedia_sp500_constituents()
if not symbols:
    symbols, resolved_source = fallback_static_sp500_universe()

selected_symbols = symbols[:MAX_FMP_SYMBOLS]
FMP_SYMBOLS = ",".join(selected_symbols)
DATA_SOURCE = "fmp"
DATA_MODE = "prices"
N_ASSETS = min(N_ASSETS, len(selected_symbols))

CONFIG["fmp_symbols"] = FMP_SYMBOLS
CONFIG["n_assets"] = N_ASSETS
CONFIG["universe_source"] = UNIVERSE_SOURCE
CONFIG["universe_source_resolved"] = resolved_source
CONFIG["max_fmp_symbols"] = MAX_FMP_SYMBOLS
CONFIG["universe_count"] = len(selected_symbols)

print(f"Universe selected: {len(selected_symbols)} stocks from {resolved_source}")
print(selected_symbols[:60])

In [ ]:
#@title Universe sanity check and sector metadata
selected_symbols = [s.strip().upper() for s in FMP_SYMBOLS.split(",") if s.strip()]
assert selected_symbols, "Empty universe"
assert len(selected_symbols) == len(set(selected_symbols)), "Duplicate symbols in universe"

SECTOR_MAP = {symbol: "Unknown" for symbol in selected_symbols}
try:
    sector_frame = pd.read_csv(DATAHUB_SP500_URL)
    sector_frame["Symbol"] = sector_frame["Symbol"].astype(str).str.upper().str.replace(".", "-", regex=False)
    sector_col = "GICS Sector" if "GICS Sector" in sector_frame.columns else None
    if sector_col:
        SECTOR_MAP.update(dict(zip(sector_frame["Symbol"], sector_frame[sector_col].astype(str))))
except Exception as exc:
    print("Sector metadata fallback: all Unknown:", sanitize_error_message(exc) if "sanitize_error_message" in globals() else str(exc)[:160])

sector_counts = pd.Series([SECTOR_MAP.get(s, "Unknown") for s in selected_symbols], name="sector").value_counts()
print(f"Active FMP universe: {len(selected_symbols)} symbols")
print("First 80 symbols:", selected_symbols[:80])
display(sector_counts.to_frame("symbols"))


## 2. Load Data

The loader is deliberately forgiving:
- detects a date column if present;
- keeps numeric columns;
- drops assets with too many missing observations;
- selects the most liquid/complete assets by variance and coverage;
- builds log returns if given prices.

For FMP, use either:
- Colab secret named `FMP_API_KEY` via the key icon in the left sidebar; or
- paste the key when prompted. It is not written to Drive.
        


In [ ]:
#@title FMP downloader with per-symbol Drive cache
def get_fmp_api_key():
    key = os.environ.get("FMP_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = (userdata.get("FMP_API_KEY") or "").strip()
        if key:
            os.environ["FMP_API_KEY"] = key
            return key
    except Exception:
        pass
    key = getpass.getpass("Paste FMP API key (input hidden): ").strip()
    os.environ["FMP_API_KEY"] = key
    return key

def sanitize_fmp_error(exc, api_key=None):
    text = str(exc)
    if api_key:
        text = text.replace(api_key, "***")
    text = re.sub(r"apikey=[^&\s]+", "apikey=***", text)
    return text[:300]

def parse_fmp_payload(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and isinstance(payload.get("historical"), list):
        return payload["historical"]
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return []

def fetch_fmp_symbol(symbol, api_key, start_date, end_date=None):
    symbol = symbol.strip().upper()
    params = {"symbol": symbol, "from": start_date, "apikey": api_key}
    if end_date:
        params["to"] = end_date

    stable_url = "https://financialmodelingprep.com/stable/historical-price-eod/full"
    response = requests.get(stable_url, params=params, timeout=30)
    rows = parse_fmp_payload(response.json()) if response.ok else []

    if not rows:
        legacy_url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
        legacy_params = {"from": start_date, "apikey": api_key}
        if end_date:
            legacy_params["to"] = end_date
        legacy_response = requests.get(legacy_url, params=legacy_params, timeout=30)
        legacy_response.raise_for_status()
        rows = parse_fmp_payload(legacy_response.json())

    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError(f"No FMP rows returned for {symbol}")
    if "date" not in frame.columns:
        raise RuntimeError(f"FMP response for {symbol} has no date column")

    price_col = "adjClose" if "adjClose" in frame.columns else "close"
    if price_col not in frame.columns:
        raise RuntimeError(f"FMP response for {symbol} has no close/adjClose column")

    out = frame[["date", price_col]].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out[price_col] = pd.to_numeric(out[price_col], errors="coerce")
    out = out.dropna().sort_values("date").drop_duplicates("date")
    out = out.rename(columns={price_col: symbol}).set_index("date")
    return out

def download_fmp_prices(symbols, start_date, end_date=None, sleep_seconds=0.12, force_refresh=False):
    api_key = get_fmp_api_key()
    symbol_cache_dir = DATA_DIR / "fmp_symbol_cache"
    symbol_cache_dir.mkdir(parents=True, exist_ok=True)

    frames = []
    errors = {}

    for symbol in tqdm(symbols, desc="FMP download/cache"):
        cache_path = symbol_cache_dir / f"{symbol}_{start_date}_{end_date or 'today'}.csv"
        try:
            if cache_path.exists() and not force_refresh:
                frame = pd.read_csv(cache_path, parse_dates=["date"]).set_index("date")
            else:
                frame = fetch_fmp_symbol(symbol, api_key, start_date, end_date)
                frame.reset_index(names="date").to_csv(cache_path, index=False)
            frames.append(frame)
        except Exception as exc:
            errors[symbol] = sanitize_fmp_error(exc, api_key)
        time.sleep(float(sleep_seconds))

    if not frames:
        raise RuntimeError(f"FMP download returned no usable symbols. Errors: {errors}")

    prices = pd.concat(frames, axis=1).sort_index().ffill(limit=5).dropna(axis=0, how="all")
    min_obs = max(252, int(len(prices) * MIN_HISTORY_FRACTION))
    prices = prices.dropna(axis=1, thresh=min_obs)
    if errors:
        print("Skipped symbols:", errors)
    print(f"Usable price columns: {prices.shape[1]} / requested {len(symbols)}")
    return prices


In [ ]:
#@title Load data with Drive cache: FMP, CSV, or demo
symbol_count = len([s for s in FMP_SYMBOLS.split(",") if s.strip()])
FMP_CACHE_NAME = f"fmp_prices_cache_v2_{FMP_START_DATE}_{FMP_END_DATE or 'today'}_{symbol_count}.csv"
FMP_CACHE_PATH = DATA_DIR / FMP_CACHE_NAME
FORCE_FMP_REFRESH = False  # True only when you intentionally want to redownload.

def make_demo_prices(path: Path, n_days=2400, n_assets=80):
    rng = np.random.default_rng(SEED)
    dates = pd.bdate_range("2015-01-01", periods=n_days)
    market = rng.normal(0.00025, 0.010, size=n_days)
    crisis = rng.random(n_days) < 0.025
    market[crisis] += rng.normal(-0.035, 0.025, size=crisis.sum())
    prices = {}
    for i in range(n_assets):
        beta = rng.uniform(0.6, 1.5)
        sector_shock = rng.normal(0, 0.006, size=n_days)
        idio = rng.normal(0, rng.uniform(0.006, 0.018), size=n_days)
        rets = beta * market + 0.35 * sector_shock + idio
        prices[f"ASSET_{i:03d}"] = 100 * np.exp(np.cumsum(rets))
    demo = pd.DataFrame(prices, index=dates).reset_index(names="date")
    demo.to_csv(path, index=False)
    return path

def load_cached_prices(path):
    frame = pd.read_csv(path)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame = frame.dropna(subset=["date"]).sort_values("date")
    return frame

if DATA_SOURCE == "fmp":
    symbols = [s.strip().upper() for s in FMP_SYMBOLS.split(",") if s.strip()]
    if not symbols:
        raise ValueError("FMP_SYMBOLS is empty")

    if FMP_CACHE_PATH.exists() and not FORCE_FMP_REFRESH:
        print("Using aggregate Drive cache:", FMP_CACHE_PATH)
        raw = load_cached_prices(FMP_CACHE_PATH)
    else:
        print("Building aggregate cache from FMP/per-symbol cache...")
        prices = download_fmp_prices(
            symbols,
            FMP_START_DATE,
            FMP_END_DATE or None,
            FMP_SLEEP_SECONDS,
            force_refresh=FORCE_FMP_REFRESH,
        )
        raw = prices.reset_index(names="date")
        raw.to_csv(FMP_CACHE_PATH, index=False)
        print("Aggregate cache saved:", FMP_CACHE_PATH)

    CSV_PATH = FMP_CACHE_PATH
    DATA_MODE = "prices"

elif DATA_SOURCE == "demo":
    CSV_PATH = make_demo_prices(Path(CSV_PATH))
    raw = pd.read_csv(CSV_PATH)
    DATA_MODE = "prices"

elif DATA_SOURCE == "csv":
    if not Path(CSV_PATH).exists():
        raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
    raw = pd.read_csv(CSV_PATH)

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

print(raw.shape)
display(raw.head())


In [ ]:
#@title Prepare return matrix
def prepare_returns(frame: pd.DataFrame, mode: str, n_assets: int) -> pd.DataFrame:
    frame = frame.copy()
    date_col = None
    for col in frame.columns:
        if str(col).lower() in {"date", "datetime", "timestamp"}:
            date_col = col
            break
    if date_col is not None:
        frame[date_col] = pd.to_datetime(frame[date_col], errors="coerce")
        frame = frame.sort_values(date_col).set_index(date_col)
    else:
        frame.index = pd.RangeIndex(len(frame))

    numeric = frame.apply(pd.to_numeric, errors="coerce").sort_index()
    numeric = numeric.ffill(limit=5)
    min_obs = max(252, int(len(numeric) * MIN_HISTORY_FRACTION))
    numeric = numeric.dropna(axis=1, thresh=min_obs)

    if mode == "prices":
        numeric = numeric.loc[:, (numeric > 0).all(axis=0)]
        returns = np.log(numeric).diff()
    else:
        returns = numeric.copy()

    returns = returns.replace([np.inf, -np.inf], np.nan)
    ret_min_obs = max(252, int(len(returns) * MIN_HISTORY_FRACTION))
    returns = returns.dropna(axis=1, thresh=ret_min_obs)

    vol = returns.std(skipna=True)
    valid_cols = vol[(vol > 1e-5) & (vol < 0.25)].index.tolist()
    preferred_order = [s.strip().upper() for s in FMP_SYMBOLS.split(",") if s.strip()]
    ordered = [col for col in preferred_order if col in valid_cols]
    extras = [col for col in valid_cols if col not in ordered]
    keep = (ordered + extras)[:n_assets]

    returns = returns[keep].ffill(limit=5).dropna(axis=0, how="any")

    # V8: preserve real crash tails by default. Only obvious bad prints are neutralized.
    tail_audit = {
        "mode": RETURN_CLIP_MODE,
        "pre_clip_min": float(np.nanmin(returns.values)),
        "pre_clip_max": float(np.nanmax(returns.values)),
        "bad_print_abs_limit": float(BAD_PRINT_ABS_RETURN_LIMIT),
    }
    if RETURN_CLIP_MODE == "wide_quantile":
        lower = returns.quantile(RETURN_CLIP_LOWER_Q)
        upper = returns.quantile(RETURN_CLIP_UPPER_Q)
        returns = returns.clip(lower=lower, upper=upper, axis=1)
        tail_audit["quantile_clip"] = [float(RETURN_CLIP_LOWER_Q), float(RETURN_CLIP_UPPER_Q)]
    elif RETURN_CLIP_MODE == "bad_print_only":
        bad_print_mask = returns.abs() > BAD_PRINT_ABS_RETURN_LIMIT
        tail_audit["bad_print_values_replaced"] = int(bad_print_mask.sum().sum())
        returns = returns.mask(bad_print_mask).ffill(limit=1).dropna(axis=0, how="any")
    elif RETURN_CLIP_MODE != "none":
        raise ValueError(f"Unknown RETURN_CLIP_MODE={RETURN_CLIP_MODE}")
    tail_audit["post_clip_min"] = float(np.nanmin(returns.values))
    tail_audit["post_clip_max"] = float(np.nanmax(returns.values))
    CONFIG["tail_preservation_audit"] = tail_audit
    returns = returns.astype("float32")

    if returns.empty or returns.shape[1] < min(50, n_assets):
        raise RuntimeError(f"Return matrix too small after filtering: {returns.shape}")
    return returns

returns_df = prepare_returns(raw, DATA_MODE, N_ASSETS)
print("Return matrix:", returns_df.shape)
display(returns_df.head())
returns_df.describe().T.head()

print("Tail preservation audit:", json.dumps(CONFIG.get("tail_preservation_audit", {}), indent=2))


## 3. Regime Detection

Regime labels are used for classifier-free guidance. The model learns both:
- unconditional denoising;
- conditional denoising for bull/bear/crisis/recovery-like clusters.

We use a simple KMeans regime detector over rolling volatility, mean return, drawdown, and average correlation. You can replace this cell with an HMM or your own macro labels.
        


In [ ]:
#@title Regime labels: train-only HMM/KMeans with robust macro/VIX/rates fallback

def rolling_regime_features(returns: pd.DataFrame, lookback=60):
    port = returns.mean(axis=1)
    roll_mean = port.rolling(lookback, min_periods=max(20, lookback // 2)).mean()
    roll_vol = port.rolling(lookback, min_periods=max(20, lookback // 2)).std()

    eq = (1 + port.fillna(0)).cumprod()
    roll_peak = eq.rolling(lookback, min_periods=max(20, lookback // 2)).max()
    drawdown = eq / roll_peak - 1

    # Fast average-correlation proxy. Avoids the huge rolling corr matrix.
    n = returns.shape[1]
    asset_var = returns.rolling(lookback, min_periods=max(20, lookback // 2)).var().mean(axis=1)
    market_var = port.rolling(lookback, min_periods=max(20, lookback // 2)).var()
    avg_corr = ((n * market_var / asset_var.clip(lower=1e-12)) - 1) / max(n - 1, 1)
    avg_corr = avg_corr.clip(-1, 1)

    feats = pd.DataFrame({
        "mean": roll_mean,
        "vol": roll_vol,
        "drawdown": drawdown,
        "avg_corr": avg_corr,
    }, index=returns.index).replace([np.inf, -np.inf], np.nan)

    return feats


def compute_temporal_split_idx(index):
    """Return the first out-of-sample row. The row at split_idx is validation."""
    n = len(index)
    min_train = max(N_REGIMES * 20, WINDOW_SIZE + 252)

    if n <= min_train + WINDOW_SIZE:
        return max(1, int(n * TRAIN_FRACTION))

    if VALIDATION_MODE == "walk_forward_crisis":
        requested = int(np.searchsorted(index.values, np.datetime64(pd.Timestamp(WALK_FORWARD_TRAIN_END))))
        split_idx = max(requested, min_train)
        split_idx = min(split_idx, n - WINDOW_SIZE)
        return int(split_idx)

    return max(min_train, int(n * TRAIN_FRACTION))


def safe_yf_download_one(ticker, start, end):
    if not YFINANCE_AVAILABLE:
        return pd.DataFrame()

    for attempt in range(3):
        try:
            raw = yf.download(
                ticker,
                start=start,
                end=end,
                auto_adjust=False,
                progress=False,
                threads=False,
            )
            if raw is not None and len(raw):
                raw = raw.copy()
                raw.index = pd.to_datetime(raw.index)
                return raw
        except Exception as exc:
            print(f"yfinance failed for {ticker}, attempt {attempt + 1}/3:", str(exc)[:160])
            time.sleep(1.5 * (attempt + 1))

    print(f"Skipping optional macro ticker after retries: {ticker}")
    return pd.DataFrame()


def fetch_macro_context(index):
    if not USE_MACRO_CONDITIONING or not YFINANCE_AVAILABLE:
        return pd.DataFrame(index=index)

    start = (pd.Timestamp(index.min()) - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    end = (pd.Timestamp(index.max()) + pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    out = pd.DataFrame(index=pd.DatetimeIndex(index))

    vix = safe_yf_download_one("^VIX", start, end)
    if len(vix) and "Close" in vix:
        s = vix["Close"].astype(float)
        s.name = "vix"
        out["vix"] = s.reindex(out.index).ffill()
        out["vix_change_5d"] = out["vix"].pct_change(5)

    tnx = safe_yf_download_one("^TNX", start, end)
    if len(tnx) and "Close" in tnx:
        s = (tnx["Close"].astype(float) / 100.0)
        s.name = "tnx_10y_pct"
        out["tnx_10y_pct"] = s.reindex(out.index).ffill()
        out["tnx_change_20d"] = out["tnx_10y_pct"].diff(20)

    spy = safe_yf_download_one("SPY", start, end)
    if len(spy) and "Volume" in spy:
        s = np.log1p(spy["Volume"].astype(float))
        s.name = "spy_volume_log"
        out["spy_volume_log"] = s.reindex(out.index).ffill()
        vol_std = out["spy_volume_log"].rolling(60, min_periods=20).std()
        out["spy_volume_z60"] = (
            out["spy_volume_log"] - out["spy_volume_log"].rolling(60, min_periods=20).mean()
        ) / vol_std.replace(0, np.nan)

    out = out.replace([np.inf, -np.inf], np.nan).ffill()
    return out


def fit_regime_model(X_train, method):
    method_used = method

    if method == "hmm" and HMM_AVAILABLE:
        try:
            hmm = GaussianHMM(
                n_components=N_REGIMES,
                covariance_type="diag",
                n_iter=500,
                tol=1e-4,
                random_state=SEED,
            )
            hmm.fit(X_train)
            labels_train = hmm.predict(X_train)
            return labels_train, "hmm", hmm
        except Exception as exc:
            print("HMM regime fit failed; falling back to KMeans:", str(exc)[:220])
            method_used = "kmeans"

    kmeans = KMeans(n_clusters=N_REGIMES, random_state=SEED, n_init=30)
    labels_train = kmeans.fit_predict(X_train)
    return labels_train, method_used, kmeans


def predict_regime_labels(model, X, method):
    if len(X) == 0:
        return np.array([], dtype=np.int64)
    if method == "hmm":
        return model.predict(X).astype(np.int64)
    return model.predict(X).astype(np.int64)


def regime_profile_from(features_frame, labels_series):
    joined = features_frame.join(labels_series.rename("regime"), how="inner")
    profile = joined.groupby("regime").agg(
        mean_return=("mean", "mean"),
        volatility=("vol", "mean"),
        drawdown=("drawdown", "mean"),
        avg_corr=("avg_corr", "mean"),
        count=("mean", "count"),
    )

    if "vix" in joined.columns:
        profile["vix"] = joined.groupby("regime")["vix"].mean()
    if "tnx_10y_pct" in joined.columns:
        profile["tnx_10y_pct"] = joined.groupby("regime")["tnx_10y_pct"].mean()
    if "spy_volume_z60" in joined.columns:
        profile["spy_volume_z60"] = joined.groupby("regime")["spy_volume_z60"].mean()

    profile["crisis_score"] = (
        profile["volatility"].rank(pct=True)
        + (-profile["drawdown"]).rank(pct=True)
        + profile["avg_corr"].rank(pct=True)
    )

    if "vix" in profile:
        profile["crisis_score"] += profile["vix"].rank(pct=True) * 0.50
    if "spy_volume_z60" in profile:
        profile["crisis_score"] += profile["spy_volume_z60"].rank(pct=True) * 0.25

    return profile.sort_values("crisis_score", ascending=False)


features_core = rolling_regime_features(returns_df)
macro_external = fetch_macro_context(features_core.index)

# Important: only core market features are mandatory.
# Optional macro columns must never erase the whole feature frame.
macro_features_raw = features_core.join(macro_external, how="left")
macro_features_raw = macro_features_raw.replace([np.inf, -np.inf], np.nan).sort_index()

core_cols = ["mean", "vol", "drawdown", "avg_corr"]
macro_cols = [
    "vix", "vix_change_5d",
    "tnx_10y_pct", "tnx_change_20d",
    "spy_volume_log", "spy_volume_z60",
]
available_macro_cols = [c for c in macro_cols if c in macro_features_raw.columns]

features = macro_features_raw[core_cols + available_macro_cols].copy()
features[available_macro_cols] = features[available_macro_cols].ffill()

# Drop rows only if core market features are missing.
features = features.dropna(subset=core_cols)

# Optional macro features get neutral train-safe fills.
for col in available_macro_cols:
    if features[col].isna().all():
        features = features.drop(columns=[col])
    else:
        features[col] = features[col].fillna(features[col].median())

regime_feature_cols = list(features.columns)

if len(features) == 0:
    raise RuntimeError(
        "Regime features are empty after robust fallback. "
        "Check returns_df shape and date index before this cell."
    )

regime_split_idx = compute_temporal_split_idx(features.index)
regime_train_features = features.iloc[:regime_split_idx]
regime_valid_features = features.iloc[regime_split_idx:]

min_train_rows = max(N_REGIMES * 20, WINDOW_SIZE + 120)
if len(regime_train_features) < min_train_rows:
    print(
        f"Warning: only {len(regime_train_features)} train rows for regime fitting; "
        f"using all available rows before validation split fallback."
    )
    regime_split_idx = max(1, min(len(features) - 1, int(len(features) * TRAIN_FRACTION)))
    regime_train_features = features.iloc[:regime_split_idx]
    regime_valid_features = features.iloc[regime_split_idx:]

if len(regime_train_features) < N_REGIMES * 10:
    raise RuntimeError(f"Still not enough train-only rows for regime fitting: {len(regime_train_features)}")

scaler = StandardScaler()
X_regime_train = scaler.fit_transform(regime_train_features)
X_regime_valid = (
    scaler.transform(regime_valid_features)
    if len(regime_valid_features)
    else np.empty((0, len(regime_feature_cols)))
)

labels_train, RESOLVED_REGIME_METHOD, regime_model = fit_regime_model(X_regime_train, REGIME_METHOD)
labels_valid = predict_regime_labels(regime_model, X_regime_valid, RESOLVED_REGIME_METHOD)
labels = np.concatenate([labels_train, labels_valid]).astype(np.int64)

regime_series = pd.Series(labels, index=features.index, name="regime")

CONFIG["regime_method_resolved"] = RESOLVED_REGIME_METHOD
CONFIG["regime_feature_cols"] = regime_feature_cols
CONFIG["regime_fit_scope"] = "train_only"
CONFIG["regime_train_start"] = str(regime_train_features.index.min().date())
CONFIG["regime_train_end"] = str(regime_train_features.index.max().date())
CONFIG["regime_valid_start"] = str(regime_valid_features.index.min().date()) if len(regime_valid_features) else None
CONFIG["regime_validation_labels"] = "predicted_by_train_fitted_model"
CONFIG["macro_optional_missing_tolerated"] = True

train_regime_series = regime_series.iloc[:regime_split_idx]
summary_train = regime_train_features.join(train_regime_series).groupby("regime").agg(["mean", "std", "count"])
print("Train-only regime feature summary:")
display(summary_train)

regime_profile_train = regime_profile_from(regime_train_features, train_regime_series)
print("Train-only regime crisis profile:")
display(regime_profile_train)

if AUTO_TARGET_CRISIS_REGIME:
    TARGET_REGIME = int(regime_profile_train["crisis_score"].idxmax())
    CONFIG["target_regime"] = TARGET_REGIME
    print("Auto-selected train-only crisis-like target regime:", TARGET_REGIME)

returns_aligned = returns_df.loc[regime_series.index]
macro_features_aligned = features.loc[returns_aligned.index].astype("float32")
MACRO_FEATURE_COLUMNS = list(macro_features_aligned.columns)
CONFIG["macro_feature_columns"] = MACRO_FEATURE_COLUMNS

print("Aligned returns:", returns_aligned.shape)
print("Macro conditioning features:", MACRO_FEATURE_COLUMNS)
print("Regime fit rows:", len(regime_train_features), "| Out-of-sample labeled rows:", len(regime_valid_features))

## 4. Dataset

Each training example is a rolling window of normalized multi-asset returns:

`x_0.shape = (window_size, n_assets)`

The DDPM forward process adds noise to that entire matrix; the score network learns to predict the noise.
        


In [ ]:
#@title Factor engine, datasets, and temporal split
class FactorWindowDataset(Dataset):
    def __init__(self, factor_frame: pd.DataFrame, regimes: pd.Series, macro_features: pd.DataFrame, window_size: int, factor_mu=None, factor_sigma=None, macro_mu=None, macro_sigma=None, label_start_date=None, asset_columns=None):
        self.factor_columns = list(factor_frame.columns)
        self.columns = list(asset_columns) if asset_columns is not None else []
        self.index = factor_frame.index
        x_raw = torch.tensor(factor_frame.values, dtype=torch.float32)
        self.factor_mu = x_raw.mean(dim=0) if factor_mu is None else factor_mu
        self.factor_sigma = x_raw.std(dim=0).clamp_min(1e-6) if factor_sigma is None else factor_sigma
        self.x = (x_raw - self.factor_mu) / self.factor_sigma
        self.factor_raw = x_raw
        macro = torch.tensor(macro_features.loc[factor_frame.index].values, dtype=torch.float32)
        self.macro_columns = list(macro_features.columns)
        self.macro_mu = macro.mean(dim=0) if macro_mu is None else macro_mu
        self.macro_sigma = macro.std(dim=0).clamp_min(1e-6) if macro_sigma is None else macro_sigma
        self.macro = torch.nan_to_num((macro - self.macro_mu) / self.macro_sigma, nan=0.0, posinf=0.0, neginf=0.0)
        self.regimes = torch.tensor(regimes.loc[factor_frame.index].values, dtype=torch.long)
        self.window_size = int(window_size)
        self.label_start_date = pd.Timestamp(label_start_date) if label_start_date is not None else None
        self.window_end_dates = pd.Index(self.index[self.window_size - 1:])

    def __len__(self):
        return max(0, len(self.x) - self.window_size + 1)

    def __getitem__(self, idx):
        end = idx + self.window_size
        return self.x[idx:end], self.macro[idx:end], self.regimes[end - 1]


class AssetWindowDataset(Dataset):
    def __init__(self, returns: pd.DataFrame, regimes: pd.Series, window_size: int):
        self.columns = list(returns.columns)
        self.index = returns.index
        self.x = torch.tensor(returns.values, dtype=torch.float32)
        self.regimes = torch.tensor(regimes.loc[returns.index].values, dtype=torch.long)
        self.window_size = int(window_size)
        self.window_end_dates = pd.Index(self.index[self.window_size - 1:])

    def __len__(self):
        return max(0, len(self.x) - self.window_size + 1)

    def __getitem__(self, idx):
        end = idx + self.window_size
        return self.x[idx:end], self.regimes[end - 1]


def sanitize_factor_name(x):
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(x)).strip("_")[:40] or "unknown"


split_idx = compute_temporal_split_idx(returns_aligned.index)
valid_context_start_idx = max(0, split_idx - WINDOW_SIZE + 1)
valid_label_start = returns_aligned.index[split_idx]

train_returns = returns_aligned.iloc[:split_idx]
valid_returns = returns_aligned.iloc[valid_context_start_idx:]
train_regimes = regime_series.loc[train_returns.index]
valid_regimes = regime_series.loc[valid_returns.index]
train_macro = macro_features_aligned.loc[train_returns.index]
valid_macro = macro_features_aligned.loc[valid_returns.index]
asset_columns = list(train_returns.columns)

asset_mu_np = train_returns[asset_columns].mean(axis=0).values.astype(np.float32)
asset_sigma_np = train_returns[asset_columns].std(axis=0).replace(0, np.nan).fillna(1e-6).values.astype(np.float32)
asset_sigma_np = np.clip(asset_sigma_np, 1e-6, None)
X_train_std = ((train_returns[asset_columns].values.astype(np.float32) - asset_mu_np) / asset_sigma_np).astype(np.float32)

N_PCA_FACTORS = int(min(FACTOR_PCA_COMPONENTS, X_train_std.shape[0] - 2, X_train_std.shape[1] - 1))
pca = PCA(n_components=N_PCA_FACTORS, svd_solver="randomized", random_state=SEED)
pca.fit(X_train_std)

asset_sectors = pd.Series([SECTOR_MAP.get(c, "Unknown") if "SECTOR_MAP" in globals() else "Unknown" for c in asset_columns], index=asset_columns)
sector_names = sorted(asset_sectors.fillna("Unknown").unique().tolist())
sector_matrix = np.zeros((len(asset_columns), len(sector_names)), dtype=np.float32)
for j, sector in enumerate(sector_names):
    idx = np.where(asset_sectors.values == sector)[0]
    if len(idx):
        sector_matrix[idx, j] = 1.0 / len(idx)


def compute_factor_frame(returns_frame: pd.DataFrame) -> pd.DataFrame:
    X = returns_frame[asset_columns].values.astype(np.float32)
    market = X.mean(axis=1, keepdims=True)
    sector = X @ sector_matrix
    X_std = ((X - asset_mu_np) / asset_sigma_np).astype(np.float32)
    pca_scores = pca.transform(X_std).astype(np.float32)
    values = np.concatenate([market, sector, pca_scores], axis=1).astype(np.float32)
    columns = ["market"] + [f"sector_{sanitize_factor_name(s)}" for s in sector_names] + [f"pca_{i+1:02d}" for i in range(N_PCA_FACTORS)]
    return pd.DataFrame(values, index=returns_frame.index, columns=columns)


FACTOR_RAW_ALIGNED = compute_factor_frame(returns_aligned)
train_factor = FACTOR_RAW_ALIGNED.loc[train_returns.index]
valid_factor = FACTOR_RAW_ALIGNED.loc[valid_returns.index]
FACTOR_COLUMNS = list(train_factor.columns)
N_FACTORS_ACTUAL = len(FACTOR_COLUMNS)

F_train = train_factor.values.astype(np.float64)
R_train = train_returns[asset_columns].values.astype(np.float64)
F_aug = np.concatenate([np.ones((len(F_train), 1)), F_train], axis=1)
ridge = np.eye(F_aug.shape[1]) * float(FACTOR_RIDGE_ALPHA)
ridge[0, 0] = 0.0
RECON_BETA = np.linalg.solve(F_aug.T @ F_aug + ridge, F_aug.T @ R_train).astype(np.float32)


def reconstruct_mean_np(factor_values):
    f = np.asarray(factor_values, dtype=np.float32)
    aug = np.concatenate([np.ones((*f.shape[:-1], 1), dtype=np.float32), f], axis=-1)
    return np.matmul(aug, RECON_BETA).astype(np.float32)


mean_recon_all = reconstruct_mean_np(FACTOR_RAW_ALIGNED.values.astype(np.float32))
RESIDUAL_RETURNS_ALIGNED = pd.DataFrame(
    returns_aligned[asset_columns].values.astype(np.float32) - mean_recon_all,
    index=returns_aligned.index,
    columns=asset_columns,
)
resid_train = RESIDUAL_RETURNS_ALIGNED.loc[train_returns.index].values.astype(np.float32)
resid_cov = np.cov(resid_train.T)
resid_diag = np.diag(np.diag(resid_cov))
resid_cov = (1 - RESIDUAL_COV_SHRINKAGE) * resid_cov + RESIDUAL_COV_SHRINKAGE * resid_diag + 1e-7 * np.eye(resid_cov.shape[0])

def safe_cholesky_np(mat):
    jitter = 1e-7
    eye = np.eye(mat.shape[0])
    for _ in range(8):
        try:
            return np.linalg.cholesky(mat + jitter * eye).astype(np.float32)
        except np.linalg.LinAlgError:
            jitter *= 10
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, 1e-8, None)
    return (vecs @ np.diag(np.sqrt(vals))).astype(np.float32)

RESIDUAL_COV_CHOL = safe_cholesky_np(resid_cov)

train_ds = FactorWindowDataset(train_factor, train_regimes, train_macro, WINDOW_SIZE, asset_columns=asset_columns)
valid_ds = FactorWindowDataset(
    valid_factor, valid_regimes, valid_macro, WINDOW_SIZE,
    train_ds.factor_mu, train_ds.factor_sigma, train_ds.macro_mu, train_ds.macro_sigma,
    label_start_date=valid_label_start, asset_columns=asset_columns,
)
asset_train_ds = AssetWindowDataset(train_returns, train_regimes, WINDOW_SIZE)
asset_valid_ds = AssetWindowDataset(valid_returns, valid_regimes, WINDOW_SIZE)

if len(valid_ds) and valid_ds.window_end_dates.min() < valid_label_start:
    raise RuntimeError("Validation windows include a pre-cutoff label date; split logic is leaking train labels into validation.")

N_ASSETS_ACTUAL = len(asset_columns)
N_MACRO_FEATURES = train_ds.macro.shape[-1]
CONFIG["n_assets_actual"] = N_ASSETS_ACTUAL
CONFIG["n_factors_actual"] = N_FACTORS_ACTUAL
CONFIG["n_macro_features"] = N_MACRO_FEATURES
CONFIG["factor_columns"] = FACTOR_COLUMNS
CONFIG["sector_factor_count"] = len(sector_names)
CONFIG["pca_factor_count"] = N_PCA_FACTORS
CONFIG["pca_explained_variance_ratio_sum"] = float(np.sum(pca.explained_variance_ratio_))
CONFIG["train_start"] = str(train_returns.index.min().date())
CONFIG["train_end"] = str(train_returns.index.max().date())
CONFIG["valid_context_start"] = str(valid_returns.index.min().date())
CONFIG["valid_label_start"] = str(valid_label_start.date())
CONFIG["valid_start"] = str(valid_label_start.date())
CONFIG["valid_end"] = str(valid_returns.index.max().date())
CONFIG["validation_protocol"] = "walk-forward labels start at valid_label_start; prior rows are context only"
CONFIG["model_family"] = MODEL_FAMILY

FACTOR_MU_T = train_ds.factor_mu.to(DEVICE)
FACTOR_SIGMA_T = train_ds.factor_sigma.to(DEVICE)
RECON_BETA_T = torch.tensor(RECON_BETA, dtype=torch.float32, device=DEVICE)
ASSET_CORR_PROJECTOR = torch.randn(N_ASSETS_ACTUAL, min(CORR_PROJECTION_DIM, N_ASSETS_ACTUAL), device=DEVICE) / math.sqrt(min(CORR_PROJECTION_DIM, N_ASSETS_ACTUAL))

train_window_regimes = np.array([int(train_ds[i][2]) for i in range(len(train_ds))])
regime_counts = pd.Series(train_window_regimes).value_counts().sort_index()
print("Train window regime counts:")
display(regime_counts.to_frame("windows"))

sample_weights = None
if USE_REGIME_BALANCED_SAMPLER:
    inv = {int(k): (len(train_window_regimes) / max(v, 1)) ** REGIME_SAMPLER_POWER for k, v in regime_counts.items()}
    sample_weights = np.array([inv[int(r)] for r in train_window_regimes], dtype=np.float64)
    if AUTO_TARGET_CRISIS_REGIME:
        sample_weights[train_window_regimes == int(TARGET_REGIME)] *= TARGET_REGIME_LOSS_BOOST
    sample_weights = sample_weights / sample_weights.mean()
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    shuffle = False
else:
    sampler = None
    shuffle = True

REGIME_LOSS_WEIGHTS = torch.ones(N_REGIMES + 1, device=DEVICE)
for k, v in regime_counts.items():
    REGIME_LOSS_WEIGHTS[int(k)] = (float(regime_counts.max()) / max(float(v), 1.0)) ** REGIME_LOSS_POWER
if AUTO_TARGET_CRISIS_REGIME:
    REGIME_LOSS_WEIGHTS[int(TARGET_REGIME)] *= TARGET_REGIME_LOSS_BOOST
REGIME_LOSS_WEIGHTS[:N_REGIMES] = REGIME_LOSS_WEIGHTS[:N_REGIMES] / REGIME_LOSS_WEIGHTS[:N_REGIMES].mean().clamp_min(1e-6)
print("Regime loss weights:", {i: float(REGIME_LOSS_WEIGHTS[i].detach().cpu()) for i in range(N_REGIMES)})

pin_memory = DEVICE == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=shuffle, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=pin_memory, drop_last=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory, drop_last=False,
)

print("Train windows:", len(train_ds), "Valid windows:", len(valid_ds), "Assets:", N_ASSETS_ACTUAL, "Factors:", N_FACTORS_ACTUAL, "Macro features:", N_MACRO_FEATURES)
print("PCA explained variance:", round(CONFIG["pca_explained_variance_ratio_sum"], 4))
print("Train period:", CONFIG["train_start"], "to", CONFIG["train_end"])
print("Valid context:", CONFIG["valid_context_start"], "| Valid labels:", CONFIG["valid_label_start"], "to", CONFIG["valid_end"])


## 5. Model

Architecture choices:
- Transformer score network over return windows.
- Sinusoidal DDPM timestep embeddings.
- Regime embedding with classifier-free dropout.
- Asset embedding so the network can learn cross-sectional identity.
- Tail-weighted denoising objective.
- Correlation loss against empirical correlation matrix.
- EMA weights for sampling.
        


In [ ]:
#@title Beta schedules
def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-5, 0.02)

def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

def make_schedule(timesteps, kind):
    betas = cosine_beta_schedule(timesteps) if kind == "cosine" else linear_beta_schedule(timesteps)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    return {
        "betas": betas.to(DEVICE),
        "alphas": alphas.to(DEVICE),
        "alphas_cumprod": alphas_cumprod.to(DEVICE),
        "sqrt_alphas_cumprod": torch.sqrt(alphas_cumprod).to(DEVICE),
        "sqrt_one_minus_alphas_cumprod": torch.sqrt(1.0 - alphas_cumprod).to(DEVICE),
    }

SCHEDULE = make_schedule(N_TIMESTEPS, BETA_SCHEDULE)



In [ ]:
#@title Factor score network
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device) * -(math.log(10000) / max(half - 1, 1)))
        args = t.float()[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if emb.shape[-1] < self.dim:
            emb = F.pad(emb, (0, self.dim - emb.shape[-1]))
        return emb


class FactorDDPMScoreNet(nn.Module):
    def __init__(self, factor_dim, window_size, d_model, n_heads, n_layers, d_ff, dropout, n_regimes, macro_dim=0):
        super().__init__()
        self.factor_dim = int(factor_dim)
        self.window_size = int(window_size)
        self.d_model = int(d_model)
        self.input_proj = nn.Linear(self.factor_dim + 4, d_model)
        self.macro_dim = int(macro_dim)
        self.macro_proj = nn.Linear(self.macro_dim, d_model) if self.macro_dim > 0 else None
        self.time_embed = nn.Sequential(
            SinusoidalEmbedding(d_model),
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.regime_embed = nn.Embedding(n_regimes + 1, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, window_size, d_model) * 0.02)
        self.temporal_conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)
        self.conv_norm = nn.LayerNorm(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, self.factor_dim),
        )

    def forward(self, x_t, t, regime=None, macro=None):
        summary_feats = torch.stack(
            [
                x_t[..., 0],
                x_t.mean(dim=-1),
                x_t.std(dim=-1).clamp_max(20.0),
                x_t.abs().mean(dim=-1),
            ],
            dim=-1,
        )
        h = self.input_proj(torch.cat([x_t, summary_feats], dim=-1))
        if self.macro_proj is not None and macro is not None:
            h = h + self.macro_proj(macro.float()).to(h.dtype)
        h = h + self.pos_embed[:, :x_t.shape[1], :]
        h = self.conv_norm(h + self.temporal_conv(h.transpose(1, 2)).transpose(1, 2))
        h = h + self.time_embed(t)[:, None, :]
        if regime is None:
            r = torch.full((x_t.shape[0],), N_REGIMES, device=x_t.device, dtype=torch.long)
        else:
            r = regime
        h = h + self.regime_embed(r)[:, None, :]
        h = self.encoder(h)
        h = self.norm(h)
        return self.output(h)


model = FactorDDPMScoreNet(
    factor_dim=N_FACTORS_ACTUAL, window_size=WINDOW_SIZE, d_model=D_MODEL,
    n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT,
    n_regimes=N_REGIMES, macro_dim=N_MACRO_FEATURES,
).to(DEVICE)

sum_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {sum_params/1e6:.2f}M | factor_dim={N_FACTORS_ACTUAL} | asset_dim={N_ASSETS_ACTUAL}")


In [ ]:
#@title Trainer utilities
def extract(schedule_value, t, x_shape):
    out = schedule_value.gather(0, t)
    return out.reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))

def q_sample(x0, t, noise):
    return extract(SCHEDULE["sqrt_alphas_cumprod"], t, x0.shape) * x0 + extract(SCHEDULE["sqrt_one_minus_alphas_cumprod"], t, x0.shape) * noise

def min_snr_weights(t):
    alpha = SCHEDULE["alphas_cumprod"].gather(0, t).clamp(1e-5, 1 - 1e-5)
    snr = alpha / (1 - alpha)
    weights = torch.minimum(snr, torch.full_like(snr, MIN_SNR_GAMMA)) / snr
    return weights.clamp(0.05, 1.0)

def factors_to_raw_torch(x_norm):
    return x_norm * FACTOR_SIGMA_T[None, None, :] + FACTOR_MU_T[None, None, :]

def factor_norm_to_asset_mean_torch(x_norm):
    f_raw = factors_to_raw_torch(x_norm)
    ones = torch.ones((*f_raw.shape[:-1], 1), dtype=f_raw.dtype, device=f_raw.device)
    f_aug = torch.cat([ones, f_raw], dim=-1)
    return torch.matmul(f_aug, RECON_BETA_T.to(f_aug.dtype))

def batch_corr(x):
    flat = x.reshape(-1, x.shape[-1]).float()
    flat = flat - flat.mean(dim=0, keepdim=True)
    cov = flat.T @ flat / max(flat.shape[0] - 1, 1)
    stdv = torch.sqrt(torch.diag(cov).clamp_min(1e-8))
    corr = cov / (stdv[:, None] * stdv[None, :] + 1e-8)
    return torch.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)

def portfolio_terminal_and_drawdown(asset_returns):
    port = asset_returns.mean(dim=-1).clamp(min=-0.95, max=1.50)
    wealth = torch.cumprod(1 + port, dim=1)
    terminal = wealth[:, -1] - 1
    peaks = torch.cummax(wealth, dim=1).values
    drawdown = wealth / peaks.clamp_min(1e-8) - 1
    return terminal, drawdown.min(dim=1).values, port

def tail_weights(x0):
    factor_mag = x0.abs().amax(dim=(1, 2))
    asset_proxy = factor_norm_to_asset_mean_torch(x0).detach()
    terminal, max_dd, _ = portfolio_terminal_and_drawdown(asset_proxy)
    downside = (-terminal).relu() * 8.0 + (-max_dd).relu() * 4.0
    weights = 1.0 + TAIL_LAMBDA * torch.pow(factor_mag.clamp_min(0), TAIL_ALPHA) + downside
    return weights.clamp(max=TAIL_WEIGHT_CLIP)

def projected_asset_corr_loss(asset_pred, asset_true):
    projector = ASSET_CORR_PROJECTOR.to(asset_pred.dtype)
    pred_proj = torch.matmul(asset_pred, projector)
    true_proj = torch.matmul(asset_true, projector)
    return F.mse_loss(batch_corr(pred_proj), batch_corr(true_proj))

def portfolio_tail_loss(asset_pred, asset_true):
    term_p, dd_p, port_p = portfolio_terminal_and_drawdown(asset_pred)
    term_t, dd_t, port_t = portfolio_terminal_and_drawdown(asset_true)
    daily = F.smooth_l1_loss(port_p, port_t)
    tail = F.smooth_l1_loss((-term_p).relu(), (-term_t).relu())
    dd = F.smooth_l1_loss((-dd_p).relu(), (-dd_t).relu())
    return daily + tail + 0.5 * dd

def weighted_mean(values, weights):
    weights = weights.float().clamp_min(1e-6)
    return (values.float() * weights).sum() / weights.sum()

def lr_for_epoch(epoch):
    min_lr = LR * MIN_LR_RATIO
    if epoch <= WARMUP_EPOCHS:
        return min_lr + (LR - min_lr) * epoch / max(WARMUP_EPOCHS, 1)
    progress = (epoch - WARMUP_EPOCHS) / max(N_EPOCHS - WARMUP_EPOCHS, 1)
    return min_lr + 0.5 * (LR - min_lr) * (1 + math.cos(math.pi * progress))

def set_optimizer_lr(epoch):
    lr = lr_for_epoch(epoch)
    for group in optimizer.param_groups:
        group["lr"] = lr
    return lr

@torch.no_grad()
def update_ema(ema_model, model, decay):
    for ema_p, p in zip(ema_model.parameters(), model.parameters()):
        ema_p.data.mul_(decay).add_(p.data, alpha=1 - decay)

ema_model = FactorDDPMScoreNet(
    factor_dim=N_FACTORS_ACTUAL, window_size=WINDOW_SIZE, d_model=D_MODEL,
    n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT,
    n_regimes=N_REGIMES, macro_dim=N_MACRO_FEATURES,
).to(DEVICE)
ema_model.load_state_dict(model.state_dict())
ema_model.eval()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler_amp = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))


## 6. Train

The validation loss is denoising MSE plus correlation fidelity pressure. The best EMA checkpoint is written to Drive.
        


In [ ]:
#@title Training loop
def train_step(x0, macro, regime):
    model.train()
    x0 = x0.to(DEVICE, non_blocking=True)
    macro = macro.to(DEVICE, non_blocking=True)
    regime = regime.to(DEVICE, non_blocking=True)
    batch = x0.shape[0]
    t = torch.randint(0, N_TIMESTEPS, (batch,), device=DEVICE).long()
    noise = torch.randn_like(x0)
    x_t = q_sample(x0, t, noise)

    regime_in = regime.clone()
    drop = torch.rand(batch, device=DEVICE) < P_UNCOND
    regime_in[drop] = N_REGIMES

    optimizer.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
        pred = model(x_t, t, regime_in, macro)
        per_sample = ((pred - noise) ** 2).mean(dim=(1, 2))
        sample_weights = tail_weights(x0) * min_snr_weights(t) * REGIME_LOSS_WEIGHTS[regime].to(DEVICE)
        loss_denoise = weighted_mean(per_sample, sample_weights)
        pred_x0 = (x_t - extract(SCHEDULE["sqrt_one_minus_alphas_cumprod"], t, x_t.shape) * pred) / extract(SCHEDULE["sqrt_alphas_cumprod"], t, x_t.shape).clamp_min(1e-5)
        pred_x0 = pred_x0.clamp(-12, 12)
        asset_pred = factor_norm_to_asset_mean_torch(pred_x0)
        asset_true = factor_norm_to_asset_mean_torch(x0)
        loss_factor_corr = F.mse_loss(batch_corr(pred_x0), batch_corr(x0))
        loss_asset_corr = projected_asset_corr_loss(asset_pred, asset_true)
        loss_tail = portfolio_tail_loss(asset_pred, asset_true)
        loss = (
            loss_denoise.float()
            + FACTOR_CORR_LOSS_WEIGHT * loss_factor_corr.float()
            + ASSET_CORR_LOSS_WEIGHT * loss_asset_corr.float()
            + PORTFOLIO_TAIL_LOSS_WEIGHT * loss_tail.float()
        )

    if not torch.isfinite(loss):
        return None, None, None, None, None, "nonfinite_loss"

    scaler_amp.scale(loss).backward()
    scaler_amp.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    if not torch.isfinite(grad_norm):
        optimizer.zero_grad(set_to_none=True)
        scaler_amp.update()
        return None, None, None, None, None, "nonfinite_grad"

    scaler_amp.step(optimizer)
    scaler_amp.update()
    update_ema(ema_model, model, EMA_DECAY)
    return (
        float(loss.detach().cpu()),
        float(loss_denoise.detach().cpu()),
        float(loss_factor_corr.detach().cpu()),
        float(loss_asset_corr.detach().cpu()),
        float(loss_tail.detach().cpu()),
        None,
    )

@torch.no_grad()
def eval_validation_metrics(loader, use_ema=True, max_batches=VALID_SELECTION_MAX_BATCHES):
    net = ema_model if use_ema else model
    net.eval()
    rows = []
    for i, (x0, macro, regime) in enumerate(loader):
        if i >= max_batches:
            break
        x0 = x0.to(DEVICE)
        macro = macro.to(DEVICE)
        regime = regime.to(DEVICE)
        batch = x0.shape[0]
        t = torch.randint(0, N_TIMESTEPS, (batch,), device=DEVICE).long()
        noise = torch.randn_like(x0)
        x_t = q_sample(x0, t, noise)
        pred = net(x_t, t, regime, macro)
        loss_noise = ((pred - noise) ** 2).mean()
        pred_x0 = (x_t - extract(SCHEDULE["sqrt_one_minus_alphas_cumprod"], t, x_t.shape) * pred) / extract(SCHEDULE["sqrt_alphas_cumprod"], t, x_t.shape).clamp_min(1e-5)
        pred_x0 = pred_x0.clamp(-12, 12)
        asset_pred = factor_norm_to_asset_mean_torch(pred_x0)
        asset_true = factor_norm_to_asset_mean_torch(x0)
        loss_factor_corr = F.mse_loss(batch_corr(pred_x0), batch_corr(x0))
        loss_asset_corr = projected_asset_corr_loss(asset_pred, asset_true)
        loss_tail = portfolio_tail_loss(asset_pred, asset_true)
        selection = (
            loss_noise.float()
            + FACTOR_CORR_LOSS_WEIGHT * loss_factor_corr.float()
            + ASSET_CORR_LOSS_WEIGHT * loss_asset_corr.float()
            + PORTFOLIO_TAIL_LOSS_WEIGHT * loss_tail.float()
        )
        if torch.isfinite(selection):
            rows.append({
                "valid_noise_mse": float(loss_noise.cpu()),
                "valid_factor_corr_loss": float(loss_factor_corr.cpu()),
                "valid_asset_corr_loss": float(loss_asset_corr.cpu()),
                "valid_portfolio_tail_loss": float(loss_tail.cpu()),
                "valid_tail_composite": float(selection.cpu()),
            })
    if not rows:
        return {
            "valid_noise_mse": np.nan,
            "valid_factor_corr_loss": np.nan,
            "valid_asset_corr_loss": np.nan,
            "valid_portfolio_tail_loss": np.nan,
            "valid_tail_composite": np.nan,
        }
    return {key: float(np.mean([row[key] for row in rows])) for key in rows[0]}

best_valid = float("inf")
best_epoch = 0
epochs_without_improve = 0
history = []
start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    current_lr = set_optimizer_lr(epoch)
    train_losses, train_denoise, train_factor_corr, train_asset_corr, train_tail = [], [], [], [], []
    skipped = 0
    skip_reasons = {}
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{N_EPOCHS}", leave=False)
    for x0, macro, regime in pbar:
        loss, denoise, factor_corr, asset_corr, tail, reason = train_step(x0, macro, regime)
        if reason is not None:
            skipped += 1
            skip_reasons[reason] = skip_reasons.get(reason, 0) + 1
            pbar.set_postfix(skipped=skipped, lr=f"{current_lr:.2e}")
            continue
        train_losses.append(loss)
        train_denoise.append(denoise)
        train_factor_corr.append(factor_corr)
        train_asset_corr.append(asset_corr)
        train_tail.append(tail)
        pbar.set_postfix(loss=f"{loss:.4f}", acorr=f"{asset_corr:.4f}", tail=f"{tail:.4f}", skipped=skipped, lr=f"{current_lr:.2e}")
    valid_metrics = eval_validation_metrics(valid_loader, use_ema=True)
    valid = valid_metrics[CHECKPOINT_SELECTION_METRIC]
    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)) if train_losses else np.nan,
        "train_denoise_loss": float(np.mean(train_denoise)) if train_denoise else np.nan,
        "train_factor_corr_loss": float(np.mean(train_factor_corr)) if train_factor_corr else np.nan,
        "train_asset_corr_loss": float(np.mean(train_asset_corr)) if train_asset_corr else np.nan,
        "train_portfolio_tail_loss": float(np.mean(train_tail)) if train_tail else np.nan,
        "valid_selection_metric": CHECKPOINT_SELECTION_METRIC,
        "valid_selection_score": valid,
        "valid_noise_mse": valid_metrics.get("valid_noise_mse", np.nan),
        "valid_factor_corr_loss": valid_metrics.get("valid_factor_corr_loss", np.nan),
        "valid_asset_corr_loss": valid_metrics.get("valid_asset_corr_loss", np.nan),
        "valid_portfolio_tail_loss": valid_metrics.get("valid_portfolio_tail_loss", np.nan),
        "valid_tail_composite": valid_metrics.get("valid_tail_composite", np.nan),
        "lr": current_lr,
        "skipped_batches": skipped,
        "skip_reasons": json.dumps(skip_reasons, sort_keys=True),
    }
    history.append(row)
    print(row)

    improved = np.isfinite(valid) and valid < (best_valid - EARLY_STOP_MIN_DELTA)
    if improved:
        best_valid = valid
        best_epoch = epoch
        epochs_without_improve = 0
        ckpt = {
            "config": CONFIG,
            "columns": train_ds.columns,
            "factor_columns": train_ds.factor_columns,
            "macro_columns": train_ds.macro_columns,
            "factor_mu": train_ds.factor_mu.cpu(),
            "factor_sigma": train_ds.factor_sigma.cpu(),
            "macro_mu": train_ds.macro_mu.cpu(),
            "macro_sigma": train_ds.macro_sigma.cpu(),
            "recon_beta": torch.tensor(RECON_BETA),
            "residual_cov_chol": torch.tensor(RESIDUAL_COV_CHOL),
            "asset_mu": torch.tensor(asset_mu_np),
            "asset_sigma": torch.tensor(asset_sigma_np),
            "sector_names": sector_names,
            "model_state": model.state_dict(),
            "ema_state": ema_model.state_dict(),
            "history": history,
            "best_epoch": best_epoch,
            "best_valid": best_valid,
            "best_selection_metric": CHECKPOINT_SELECTION_METRIC,
        }
        torch.save(ckpt, CHECKPOINT_DIR / "best_factor_ddpm_market_simulator.pt")
    else:
        epochs_without_improve += 1

    if epoch >= MIN_EPOCHS_BEFORE_STOP and epochs_without_improve >= EARLY_STOP_PATIENCE:
        print(f"Early stop at epoch {epoch}. Best epoch: {best_epoch}, best valid: {best_valid:.6f}")
        break

print("Elapsed minutes:", round((time.time() - start) / 60, 2))
pd.DataFrame(history).tail()


In [ ]:
#@title Training curve and run diagnostics
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist["epoch"], hist["train_loss"], label="train total")
axes[0].plot(hist["epoch"], hist["valid_noise_mse"], label="valid noise mse")
axes[0].set_title("Training losses")
axes[0].legend()
axes[0].grid(True, alpha=0.25)

for col in ["train_denoise_loss", "train_factor_corr_loss", "train_asset_corr_loss", "train_portfolio_tail_loss"]:
    if col in hist:
        axes[1].plot(hist["epoch"], hist[col], label=col.replace("train_", ""))
axes[1].set_title("Loss components")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.25)

axes[2].plot(hist["epoch"], hist["lr"], label="lr")
axes[2].set_title("Learning rate")
axes[2].grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

finite_valid = hist["valid_noise_mse"].replace([np.inf, -np.inf], np.nan).dropna()
diagnostics = {
    "epochs": int(len(hist)),
    "first_valid_noise_mse": float(finite_valid.iloc[0]) if len(finite_valid) else np.nan,
    "best_valid_noise_mse": float(finite_valid.min()) if len(finite_valid) else np.nan,
    "last_valid_noise_mse": float(finite_valid.iloc[-1]) if len(finite_valid) else np.nan,
    "best_epoch": int(hist.loc[hist["valid_noise_mse"].idxmin(), "epoch"]) if len(finite_valid) else None,
    "valid_improvement_pct": float((finite_valid.iloc[0] - finite_valid.min()) / finite_valid.iloc[0] * 100) if len(finite_valid) else np.nan,
    "total_skipped_batches": int(hist["skipped_batches"].sum()) if "skipped_batches" in hist else 0,
}
display(pd.Series(diagnostics).to_frame("value"))


## 7. Sample Synthetic Stress Scenarios

Sampling uses DDIM-style deterministic jumps over the reverse process. `GUIDANCE_SCALE > 1` pushes samples toward the selected regime.
        


In [ ]:
#@title Factor DDIM sampler with residual bootstrap and stress ladder
def as_numpy_local(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def denormalize_factor_windows(x_norm):
    arr = as_numpy_local(x_norm).astype(np.float32)
    return arr * as_numpy_local(train_ds.factor_sigma)[None, None, :] + as_numpy_local(train_ds.factor_mu)[None, None, :]

def dataset_factor_windows(ds, max_samples=None, regime_filter=None, seed=SEED):
    idxs = list(range(len(ds)))
    if regime_filter is not None:
        idxs = [i for i in idxs if int(ds[i][2]) == int(regime_filter)]
    if not idxs:
        return torch.empty(0, WINDOW_SIZE, N_FACTORS_ACTUAL)
    rng = np.random.default_rng(seed)
    if max_samples is not None and len(idxs) > max_samples:
        idxs = rng.choice(idxs, size=max_samples, replace=False).tolist()
    return torch.stack([ds[i][0] for i in idxs])

def dataset_asset_windows(ds, max_samples=None, regime_filter=None, seed=SEED):
    idxs = list(range(len(ds)))
    if regime_filter is not None:
        idxs = [i for i in idxs if int(ds[i][1]) == int(regime_filter)]
    if not idxs:
        return torch.empty(0, WINDOW_SIZE, N_ASSETS_ACTUAL)
    rng = np.random.default_rng(seed)
    if max_samples is not None and len(idxs) > max_samples:
        idxs = rng.choice(idxs, size=max_samples, replace=False).tolist()
    return torch.stack([ds[i][0] for i in idxs])

def macro_condition_window(ds, target_regime, seed=SEED):
    # V8: sample a real macro trajectory instead of averaging dynamic windows into a static template.
    idxs = [i for i in range(len(ds)) if int(ds[i][2]) == int(target_regime)]
    if len(idxs) < 16:
        idxs = list(range(len(ds)))
    rng = np.random.default_rng(seed)
    chosen = int(rng.choice(idxs))
    CONFIG["macro_condition_source"] = "sampled_train_macro_window"
    CONFIG["macro_condition_window_end"] = str(ds.window_end_dates[chosen]) if hasattr(ds, "window_end_dates") else None
    return ds[chosen][1]

def quick_portfolio_summary(samples, weights=None):
    arr = as_numpy_local(samples)
    if len(arr) == 0:
        return {
            "mean_terminal": np.nan, "median_terminal": np.nan, "vol_daily": np.nan,
            "var5": np.nan, "cvar5": np.nan, "prob_loss": np.nan, "prob_drawdown_10pct": np.nan,
        }
    if weights is None:
        weights = np.ones(arr.shape[-1]) / arr.shape[-1]
    daily = np.clip(arr @ weights, -0.95, 1.50)
    terminal = np.prod(1 + daily, axis=1) - 1
    wealth = np.cumprod(1 + daily, axis=1)
    drawdown = wealth / np.maximum.accumulate(wealth, axis=1) - 1
    var5 = np.quantile(terminal, 0.05)
    return {
        "mean_terminal": float(np.mean(terminal)),
        "median_terminal": float(np.median(terminal)),
        "vol_daily": float(np.std(daily)),
        "var5": float(var5),
        "cvar5": float(terminal[terminal <= var5].mean()),
        "prob_loss": float((terminal < 0).mean()),
        "prob_drawdown_10pct": float((drawdown.min(axis=1) <= -0.10).mean()),
    }

def parse_float_list(text):
    return np.array([float(x.strip()) for x in str(text).split(",") if x.strip()], dtype=np.float32)

def stratified_stress_multipliers(n, weights, multipliers, seed=SEED):
    rng = np.random.default_rng(seed)
    weights = np.asarray(weights, dtype=np.float64)
    multipliers = np.asarray(multipliers, dtype=np.float32)
    weights = weights / weights.sum()
    if not STRESS_STRATIFIED_SAMPLING:
        return rng.choice(multipliers, size=int(n), replace=True, p=weights).astype(np.float32)
    raw_counts = weights * int(n)
    counts = np.floor(raw_counts).astype(int)
    remainder = int(n) - int(counts.sum())
    if remainder > 0:
        order = np.argsort(-(raw_counts - counts))
        counts[order[:remainder]] += 1
    elif remainder < 0:
        order = np.argsort(raw_counts - counts)
        for idx in order[:abs(remainder)]:
            if counts[idx] > 0:
                counts[idx] -= 1
    book = np.repeat(multipliers, counts)
    if len(book) < int(n):
        book = np.concatenate([book, rng.choice(multipliers, size=int(n) - len(book), replace=True, p=weights)])
    elif len(book) > int(n):
        book = book[: int(n)]
    rng.shuffle(book)
    return book.astype(np.float32)

def stress_ladder(factor_raw, seed=SEED):
    f = np.asarray(factor_raw, dtype=np.float32).copy()
    if not STRESS_SCENARIO_MODE:
        return f, np.ones(f.shape[0], dtype=np.float32)
    weights = parse_float_list(STRESS_MIX_WEIGHTS)
    multipliers = parse_float_list(STRESS_MIX_MULTIPLIERS)
    weights = weights / weights.sum()
    m = stratified_stress_multipliers(f.shape[0], weights, multipliers, seed=seed)
    center = train_factor.median(axis=0).values.astype(np.float32)
    scale = np.sqrt(m)[:, None, None]
    f[:, :, 1:] = center[None, None, 1:] + (f[:, :, 1:] - center[None, None, 1:]) * scale
    market_center = center[0]
    downside = f[:, :, 0] < market_center
    down_scale = (m[:, None] * DOWNSIDE_ASYMMETRY).astype(np.float32)
    up_scale = (0.85 + 0.15 / np.maximum(m[:, None], 1e-6)).astype(np.float32)
    f[:, :, 0] = np.where(
        downside,
        market_center + (f[:, :, 0] - market_center) * down_scale,
        market_center + (f[:, :, 0] - market_center) * up_scale,
    )

    # V8 crisis capsule: sparse location/scale stress, never a full-window deterministic floor.
    severe = m >= float(np.max(multipliers))
    if severe.any():
        rng = np.random.default_rng(seed + 999)
        train_market = train_factor.iloc[:, 0].astype(np.float32).values
        q_shift = float(np.quantile(train_market, SEVERE_MARKET_LOCATION_SHIFT_Q))
        for row in np.where(severe)[0]:
            n_days = int(rng.integers(SEVERE_SHOCK_MIN_DAYS, SEVERE_SHOCK_MAX_DAYS + 1))
            n_days = int(np.clip(n_days, 1, f.shape[1]))
            shock_days = rng.choice(f.shape[1], size=n_days, replace=False)
            local = f[row, shock_days, 0]
            centered = local - np.median(train_market)
            f[row, shock_days, 0] = q_shift + centered * SEVERE_MARKET_SCALE_BOOST
    CONFIG["stress_full_window_floor_applied"] = False
    CONFIG["stress_shock_mode_applied"] = STRESS_SHOCK_MODE
    return f, m

def residual_window_bank(regime_filter=None):
    residual_train = RESIDUAL_RETURNS_ALIGNED.loc[train_returns.index].values.astype(np.float32)
    reg = train_regimes.values.astype(int)
    windows = []
    for i in range(0, len(residual_train) - WINDOW_SIZE + 1):
        if regime_filter is None or int(reg[i + WINDOW_SIZE - 1]) == int(regime_filter):
            windows.append(residual_train[i:i + WINDOW_SIZE])
    if not windows:
        return np.empty((0, WINDOW_SIZE, N_ASSETS_ACTUAL), dtype=np.float32)
    return np.stack(windows).astype(np.float32)

RESIDUAL_WINDOW_BANK_TARGET = residual_window_bank(TARGET_REGIME)
RESIDUAL_WINDOW_BANK_ALL = residual_window_bank(None)

def reconstruct_asset_windows_from_factors(factor_raw, add_residual=True, stress=False, seed=SEED):
    factors = np.asarray(factor_raw, dtype=np.float32)
    if stress:
        factors, multipliers = stress_ladder(factors, seed=seed)
    else:
        multipliers = np.ones(factors.shape[0], dtype=np.float32)
    base = reconstruct_mean_np(factors)
    if add_residual:
        bank = RESIDUAL_WINDOW_BANK_TARGET if len(RESIDUAL_WINDOW_BANK_TARGET) >= 32 else RESIDUAL_WINDOW_BANK_ALL
        rng = np.random.default_rng(seed + 17)
        idx = rng.choice(len(bank), size=factors.shape[0], replace=True)
        resid = bank[idx] * float(RESIDUAL_BOOTSTRAP_SCALE) * np.sqrt(multipliers)[:, None, None]
        base = base + resid.astype(np.float32)
    return torch.tensor(np.clip(base, -0.80, 1.50), dtype=torch.float32), multipliers

@torch.no_grad()
def predict_noise_guided(net, x, t, regime, macro, guidance_scale):
    if regime is None:
        return net(x, t, regime, macro)
    cond = net(x, t, regime, macro)
    uncond = net(x, t, torch.full_like(regime, N_REGIMES), macro)
    return uncond + float(guidance_scale) * (cond - uncond)

@torch.no_grad()
def ddim_sample(net, n_samples, target_regime, steps=80, guidance_scale=2.5, eta=0.0, macro_template=None):
    net.eval()
    shape = (n_samples, WINDOW_SIZE, N_FACTORS_ACTUAL)
    x = torch.randn(shape, device=DEVICE)
    times = torch.linspace(N_TIMESTEPS - 1, 0, steps, device=DEVICE).long()
    regime = torch.full((n_samples,), int(target_regime), device=DEVICE, dtype=torch.long)
    macro = None if macro_template is None else macro_template.to(DEVICE).unsqueeze(0).repeat(n_samples, 1, 1)
    for i, t_scalar in enumerate(tqdm(times, desc="DDIM factor sampling", leave=False)):
        t = torch.full((n_samples,), int(t_scalar.item()), device=DEVICE, dtype=torch.long)
        eps = predict_noise_guided(net, x, t, regime, macro, guidance_scale)
        alpha_t = SCHEDULE["alphas_cumprod"][t_scalar]
        sqrt_alpha_t = torch.sqrt(alpha_t)
        sqrt_one_minus_t = torch.sqrt(1 - alpha_t)
        x0_pred = (x - sqrt_one_minus_t * eps) / sqrt_alpha_t.clamp_min(1e-5)
        x0_pred = x0_pred.clamp(-12, 12)
        if i == len(times) - 1:
            x = x0_pred
            break
        t_next = times[i + 1]
        alpha_next = SCHEDULE["alphas_cumprod"][t_next]
        sigma = eta * torch.sqrt((1 - alpha_next) / (1 - alpha_t) * (1 - alpha_t / alpha_next)).clamp_min(0)
        dir_xt = torch.sqrt((1 - alpha_next - sigma ** 2).clamp_min(0)) * eps
        noise = sigma * torch.randn_like(x) if eta > 0 else 0
        x = torch.sqrt(alpha_next) * x0_pred + dir_xt + noise
    return x

@torch.no_grad()
def ddim_sample_batched(net, n_samples, target_regime, steps, guidance_scale, eta, batch_size, macro_template=None):
    chunks = []
    remaining = int(n_samples)
    while remaining > 0:
        n = min(int(batch_size), remaining)
        chunks.append(ddim_sample(net, n, target_regime, steps=steps, guidance_scale=guidance_scale, eta=eta, macro_template=macro_template).detach().cpu())
        remaining -= n
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return torch.cat(chunks, dim=0)


def sym_matrix_power(mat, power, eps=1e-7):
    mat = np.asarray(mat, dtype=np.float64)
    mat = 0.5 * (mat + mat.T)
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, eps, None)
    return (vecs * (vals ** power)) @ vecs.T

def shrink_cov_np(x, shrinkage=0.10):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or len(x) < 2:
        return np.eye(x.shape[-1] if x.ndim == 2 else N_FACTORS_ACTUAL, dtype=np.float64)
    cov = np.cov(x, rowvar=False)
    diag = np.diag(np.diag(cov))
    return (1.0 - shrinkage) * cov + shrinkage * diag + 1e-7 * np.eye(cov.shape[0])

def calibrate_factor_windows(synthetic, target, alpha=FACTOR_CALIBRATION_ALPHA, shrinkage=FACTOR_CALIBRATION_SHRINKAGE):
    syn = np.asarray(synthetic, dtype=np.float32)
    tgt = np.asarray(target, dtype=np.float32)
    if len(syn) == 0 or len(tgt) < 8:
        return syn
    shape = syn.shape
    s = syn.reshape(-1, shape[-1]).astype(np.float64)
    t = tgt.reshape(-1, tgt.shape[-1]).astype(np.float64)
    mu_s = s.mean(axis=0, keepdims=True)
    mu_t = t.mean(axis=0, keepdims=True)
    cs = shrink_cov_np(s - mu_s, shrinkage=shrinkage)
    ct = shrink_cov_np(t - mu_t, shrinkage=shrinkage)
    whiten = sym_matrix_power(cs, -0.5)
    recolor = sym_matrix_power(ct, 0.5)
    matched = (s - mu_s) @ whiten @ recolor + mu_t
    blended = (1.0 - float(alpha)) * s + float(alpha) * matched
    lo = np.quantile(t, 0.001, axis=0, keepdims=True)
    hi = np.quantile(t, 0.999, axis=0, keepdims=True)
    pad = 0.50 * np.maximum(hi - lo, 1e-6)
    blended = np.clip(blended, lo - pad, hi + pad)
    return blended.reshape(shape).astype(np.float32)

def calibrate_asset_correlation_windows(samples, target, alpha=CHOLESKY_CALIBRATION_ALPHA, shrinkage=CHOLESKY_SHRINKAGE):
    """Train-only correlation transport. Preserves synthetic marginals more than full covariance matching."""
    syn = to_numpy(samples).astype(np.float32) if "to_numpy" in globals() else as_numpy_local(samples).astype(np.float32)
    tgt = to_numpy(target).astype(np.float32) if "to_numpy" in globals() else as_numpy_local(target).astype(np.float32)
    if not APPLY_CHOLESKY_CALIBRATION or len(syn) == 0 or len(tgt) < 16:
        return samples

    shape = syn.shape
    s = syn.reshape(-1, shape[-1]).astype(np.float64)
    t = tgt.reshape(-1, tgt.shape[-1]).astype(np.float64)
    mu_s = s.mean(axis=0, keepdims=True)
    sd_s = s.std(axis=0, keepdims=True) + 1e-7
    mu_t = t.mean(axis=0, keepdims=True)
    sd_t = t.std(axis=0, keepdims=True) + 1e-7

    zs = (s - mu_s) / sd_s
    zt = (t - mu_t) / sd_t
    corr_s = shrink_cov_np(zs, shrinkage=shrinkage)
    corr_t = shrink_cov_np(zt, shrinkage=shrinkage)
    matched_z = zs @ sym_matrix_power(corr_s, -0.5) @ sym_matrix_power(corr_t, 0.5)

    # Keep the DDPM/base marginal scale and tail level; transport mostly the cross-sectional dependence.
    transported = matched_z * sd_s + mu_s
    blended = (1.0 - float(alpha)) * s + float(alpha) * transported

    lo = np.quantile(s, 0.0005, axis=0, keepdims=True)
    hi = np.quantile(s, 0.9995, axis=0, keepdims=True)
    pad = 0.35 * np.maximum(hi - lo, 1e-6)
    blended = np.clip(blended, lo - pad, hi + pad).reshape(shape).astype(np.float32)
    return torch.tensor(blended, dtype=torch.float32)

target_factor_norm = dataset_factor_windows(train_ds, max_samples=GUIDANCE_SWEEP_SCENARIOS, regime_filter=TARGET_REGIME)
target_factor_source = "train_target_regime_factor_windows"
if len(target_factor_norm) < 64:
    target_factor_norm = dataset_factor_windows(train_ds, max_samples=GUIDANCE_SWEEP_SCENARIOS)
    target_factor_source = "train_all_regimes_factor_windows_fallback"
if len(target_factor_norm) < 64:
    raise RuntimeError("Not enough train factor windows for factor calibration target.")
target_factor_raw = denormalize_factor_windows(target_factor_norm)
CONFIG["factor_calibration_applied"] = True
CONFIG["factor_calibration_alpha"] = float(FACTOR_CALIBRATION_ALPHA)
CONFIG["factor_calibration_shrinkage"] = float(FACTOR_CALIBRATION_SHRINKAGE)
CONFIG["factor_calibration_target_source"] = target_factor_source
CONFIG["factor_calibration_uses_validation"] = False
print("Train factor calibration target:", target_factor_source, tuple(target_factor_raw.shape))

target_real_returns = dataset_asset_windows(asset_train_ds, max_samples=GUIDANCE_SWEEP_SCENARIOS, regime_filter=TARGET_REGIME)
target_source = "train_target_regime_asset_windows"
if len(target_real_returns) < 64:
    target_real_returns = dataset_asset_windows(asset_train_ds, max_samples=GUIDANCE_SWEEP_SCENARIOS)
    target_source = "train_all_regimes_asset_windows_fallback"
if len(target_real_returns) < 64:
    raise RuntimeError("Not enough train asset windows for guidance/calibration target.")

target_summary = quick_portfolio_summary(target_real_returns)
TARGET_MACRO_TEMPLATE = macro_condition_window(train_ds, TARGET_REGIME)
CONFIG["guidance_target_source"] = target_source
CONFIG["guidance_uses_validation"] = False
print("Train empirical guidance target:", target_source)
print(json.dumps(target_summary, indent=2))

guidance_rows = []
selected_guidance = GUIDANCE_SCALE
if AUTO_GUIDANCE_SWEEP:
    weights = np.ones(N_ASSETS_ACTUAL) / N_ASSETS_ACTUAL
    target_scale = {
        "mean_terminal": max(abs(target_summary["mean_terminal"]), 0.01),
        "vol_daily": max(abs(target_summary["vol_daily"]), 1e-4),
        "cvar5": max(abs(target_summary["cvar5"]), 0.01),
        "prob_loss": 1.0,
        "prob_drawdown_10pct": 1.0,
    }
    for g in [float(x.strip()) for x in GUIDANCE_SWEEP.split(",") if x.strip()]:
        test_norm = ddim_sample_batched(
            ema_model, GUIDANCE_SWEEP_SCENARIOS, TARGET_REGIME,
            steps=max(30, min(DDIM_STEPS, 80)), guidance_scale=g, eta=DDIM_ETA,
            batch_size=min(SAMPLE_BATCH_SIZE, GUIDANCE_SWEEP_SCENARIOS),
            macro_template=TARGET_MACRO_TEMPLATE,
        )
        test_factor_raw_uncalibrated = denormalize_factor_windows(test_norm)
        test_factor_raw = calibrate_factor_windows(test_factor_raw_uncalibrated, target_factor_raw)
        test_returns, _ = reconstruct_asset_windows_from_factors(test_factor_raw, add_residual=True, stress=False, seed=SEED + int(g * 100))
        summary = quick_portfolio_summary(test_returns, weights)
        score = (
            abs(summary["mean_terminal"] - target_summary["mean_terminal"]) / target_scale["mean_terminal"]
            + abs(summary["vol_daily"] - target_summary["vol_daily"]) / target_scale["vol_daily"]
            + 1.5 * abs(summary["cvar5"] - target_summary["cvar5"]) / target_scale["cvar5"]
            + 0.5 * abs(summary["prob_loss"] - target_summary["prob_loss"])
            + 0.5 * abs(summary["prob_drawdown_10pct"] - target_summary["prob_drawdown_10pct"])
        )
        guidance_rows.append({"guidance_scale": g, "score": float(score), **summary})
    guidance_sweep_df = pd.DataFrame(guidance_rows).sort_values("score")
    display(guidance_sweep_df)
    selected_guidance = float(guidance_sweep_df.iloc[0]["guidance_scale"])
else:
    guidance_sweep_df = pd.DataFrame()

print("Selected guidance scale:", selected_guidance)
CONFIG["selected_guidance_scale"] = selected_guidance

synthetic_factor_norm = ddim_sample_batched(
    ema_model, N_SCENARIOS, TARGET_REGIME, steps=DDIM_STEPS,
    guidance_scale=selected_guidance, eta=DDIM_ETA, batch_size=SAMPLE_BATCH_SIZE,
    macro_template=TARGET_MACRO_TEMPLATE,
)
synthetic_factor_raw_uncalibrated = denormalize_factor_windows(synthetic_factor_norm)
synthetic_factor_raw = calibrate_factor_windows(synthetic_factor_raw_uncalibrated, target_factor_raw)
synthetic_returns_base, base_stress_multipliers = reconstruct_asset_windows_from_factors(synthetic_factor_raw, add_residual=True, stress=False, seed=SEED + 101)
synthetic_returns, stress_multipliers = reconstruct_asset_windows_from_factors(synthetic_factor_raw, add_residual=True, stress=STRESS_SCENARIO_MODE, seed=SEED + 202)
synthetic_returns_base_uncalibrated = synthetic_returns_base.clone()
synthetic_returns_uncalibrated = synthetic_returns.clone()

if APPLY_CHOLESKY_CALIBRATION:
    synthetic_returns_base = calibrate_asset_correlation_windows(synthetic_returns_base, target_real_returns)
    synthetic_returns = calibrate_asset_correlation_windows(synthetic_returns, target_real_returns)
synthetic_returns_raw = synthetic_returns_base_uncalibrated.clone()

real_valid_returns = dataset_asset_windows(asset_valid_ds, max_samples=min(len(asset_valid_ds), N_SCENARIOS))
real_valid_target_returns = dataset_asset_windows(asset_valid_ds, max_samples=min(len(asset_valid_ds), N_SCENARIOS), regime_filter=TARGET_REGIME)
CONFIG["valid_all_windows_sampled"] = int(len(real_valid_returns))
CONFIG["valid_target_regime_windows_sampled"] = int(len(real_valid_target_returns))
CONFIG["cholesky_calibration_applied"] = bool(APPLY_CHOLESKY_CALIBRATION)
CONFIG["cholesky_calibration_alpha"] = float(CHOLESKY_CALIBRATION_ALPHA)
CONFIG["cholesky_shrinkage"] = float(CHOLESKY_SHRINKAGE)
CONFIG["cholesky_calibration_target_source"] = target_source
CONFIG["cholesky_uses_validation"] = False
CONFIG["residual_bootstrap_applied"] = True
CONFIG["stress_scenario_mode_applied"] = bool(STRESS_SCENARIO_MODE)
CONFIG["stress_stratified_sampling_applied"] = bool(STRESS_STRATIFIED_SAMPLING)
CONFIG["stress_multiplier_mean"] = float(np.mean(stress_multipliers))
CONFIG["stress_multiplier_q95"] = float(np.quantile(stress_multipliers, 0.95))
CONFIG["stress_multiplier_counts"] = {str(float(k)): int(v) for k, v in zip(*np.unique(stress_multipliers, return_counts=True))}

print("Synthetic stress:", tuple(synthetic_returns.shape), "Synthetic base:", tuple(synthetic_returns_base.shape), "Real valid all:", tuple(real_valid_returns.shape), "Real valid target:", tuple(real_valid_target_returns.shape))
print("Synthetic stress summary:", json.dumps(quick_portfolio_summary(synthetic_returns), indent=2))
print("Synthetic base summary:", json.dumps(quick_portfolio_summary(synthetic_returns_base), indent=2))


## 8. Evaluation

Core checks:
- distribution coverage: whether synthetic returns visit real bins;
- correlation fidelity: whether multi-asset dependence survives sampling;
- MMD: kernel distance between real and synthetic windows;
- VaR/CVaR: synthetic portfolio tail risk;
- tail contributors: which assets dominate losses.
        


In [ ]:
#@title Metrics and baselines: factor candidate vs train-only baselines
def to_numpy(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def distribution_coverage(real, syn, n_bins=50):
    real = to_numpy(real)
    syn = to_numpy(syn)
    n_assets = real.shape[-1]
    covs = []
    for i in range(n_assets):
        lo, hi = np.nanquantile(real[..., i], [0.001, 0.999])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            continue
        bins = np.linspace(lo, hi, n_bins)
        rh, _ = np.histogram(real[..., i].ravel(), bins=bins)
        sh, _ = np.histogram(syn[..., i].ravel(), bins=bins)
        total = (rh > 0).sum()
        covs.append(((rh > 0) & (sh > 0)).sum() / max(total, 1))
    return float(np.mean(covs)) if covs else np.nan

def corr_stats(real, syn):
    real = to_numpy(real).reshape(-1, to_numpy(real).shape[-1])
    syn = to_numpy(syn).reshape(-1, to_numpy(syn).shape[-1])
    real_corr = np.nan_to_num(np.corrcoef(real.T), nan=0.0)
    syn_corr = np.nan_to_num(np.corrcoef(syn.T), nan=0.0)
    mask = ~np.eye(real_corr.shape[0], dtype=bool)
    mae = float(np.abs(real_corr[mask] - syn_corr[mask]).mean())
    eig_real = np.sort(np.linalg.eigvalsh(real_corr))[-20:]
    eig_syn = np.sort(np.linalg.eigvalsh(syn_corr))[-20:]
    eig_rmse = float(np.sqrt(np.mean((eig_real - eig_syn) ** 2)))
    return {
        "correlation_fidelity": float(1 - mae),
        "corr_mae": mae,
        "corr_top20_eigen_rmse": eig_rmse,
    }

def mmd_rbf_projected(X, Y, proj_dim=128, max_n=1500, seed=SEED):
    X = to_numpy(X).reshape(to_numpy(X).shape[0], -1)
    Y = to_numpy(Y).reshape(to_numpy(Y).shape[0], -1)
    n = min(max_n, len(X), len(Y))
    X = X[:n].astype(np.float32)
    Y = Y[:n].astype(np.float32)
    rng = np.random.default_rng(seed)
    R = rng.normal(0, 1 / math.sqrt(proj_dim), size=(X.shape[1], proj_dim)).astype(np.float32)
    Xp = X @ R
    Yp = Y @ R
    mu = Xp.mean(axis=0, keepdims=True)
    sigma = Xp.std(axis=0, keepdims=True) + 1e-6
    Xp = (Xp - mu) / sigma
    Yp = (Yp - mu) / sigma

    def sqdist(A, B):
        return np.maximum((A * A).sum(1, keepdims=True) + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0)

    Z = np.vstack([Xp[: min(300, len(Xp))], Yp[: min(300, len(Yp))]])
    d_med = sqdist(Z, Z)
    med = np.median(d_med[d_med > 0])
    gamma = 1.0 / max(float(med), 1e-6)
    return float(np.exp(-gamma * sqdist(Xp, Xp)).mean() + np.exp(-gamma * sqdist(Yp, Yp)).mean() - 2 * np.exp(-gamma * sqdist(Xp, Yp)).mean())

def parse_int_list(text):
    return [int(x.strip()) for x in str(text).split(",") if x.strip()]

def mmd_rbf_projected_multi(X, Y, seeds=None, proj_dim=128, max_n=1500):
    seeds = parse_int_list(MMD_STABILITY_SEEDS) if seeds is None else list(seeds)
    vals = [
        mmd_rbf_projected(X, Y, proj_dim=proj_dim, max_n=max_n, seed=int(seed))
        for seed in seeds
    ]
    return {
        "mmd_rbf_projected_multi_mean": float(np.mean(vals)),
        "mmd_rbf_projected_multi_std": float(np.std(vals)),
        "mmd_rbf_projected_multi_min": float(np.min(vals)),
        "mmd_rbf_projected_multi_max": float(np.max(vals)),
    }

def sample_windows_np(x, n, seed=PRIMARY_EVAL_SUBSAMPLE_SEED):
    arr = to_numpy(x)
    if len(arr) <= n:
        return arr
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(arr), size=int(n), replace=False)
    return arr[idx]

def portfolio_terminal_returns(samples, weights=None):
    arr = to_numpy(samples)
    if weights is None:
        weights = np.ones(arr.shape[-1]) / arr.shape[-1]
    daily = np.clip(arr @ weights, -0.95, 1.50)
    return np.prod(1 + daily, axis=1) - 1, daily

def max_drawdown(daily_returns):
    wealth = np.cumprod(1 + daily_returns, axis=1)
    peaks = np.maximum.accumulate(wealth, axis=1)
    dd = wealth / peaks - 1
    return dd.min(axis=1)

def historical_bootstrap_windows(reference, n_samples, seed=SEED):
    arr = to_numpy(reference)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(arr), size=n_samples, replace=True)
    return arr[idx]

def strided_windows_np(windows, stride=WINDOW_SIZE):
    arr = to_numpy(windows)
    stride = max(1, int(stride))
    return arr[::stride]

def gaussian_cov_from_windows(reference, n_samples, window_size, seed=SEED):
    rng = np.random.default_rng(seed)
    ref = to_numpy(reference).astype(np.float32)
    X = ref.reshape(-1, ref.shape[-1])
    mu = X.mean(axis=0)
    cov = np.cov(X.T)
    diag = np.diag(np.diag(cov))
    cov = 0.95 * cov + 0.05 * diag
    jitter = 1e-6 * np.eye(cov.shape[0])
    L = np.linalg.cholesky(cov + jitter)
    z = rng.normal(size=(n_samples, window_size, ref.shape[-1])).astype(np.float32)
    return z @ L.T + mu

def t_copula_from_windows(reference, n_samples, window_size, df=BASELINE_T_COPULA_DF, seed=SEED):
    rng = np.random.default_rng(seed)
    ref = to_numpy(reference).astype(np.float32)
    X = ref.reshape(-1, ref.shape[-1])
    mu = X.mean(axis=0)
    cov = np.cov(X.T)
    diag = np.diag(np.diag(cov))
    cov = 0.95 * cov + 0.05 * diag
    L = np.linalg.cholesky(cov + 1e-6 * np.eye(cov.shape[0]))
    z = rng.normal(size=(n_samples, window_size, ref.shape[-1])).astype(np.float32)
    chi = rng.chisquare(df, size=(n_samples, window_size, 1)).astype(np.float32)
    return (z @ L.T) / np.sqrt(np.maximum(chi / df, 1e-6)) + mu

def filtered_historical_simulation(reference, n_samples, seed=SEED):
    rng = np.random.default_rng(seed)
    ref = to_numpy(reference).astype(np.float32)
    flat = ref.reshape(-1, ref.shape[-1])
    lam = float(FHS_EWMA_LAMBDA)
    vol = np.zeros_like(flat)
    vol[0] = np.nanstd(flat, axis=0) + 1e-6
    for i in range(1, len(flat)):
        vol[i] = np.sqrt(lam * vol[i - 1] ** 2 + (1 - lam) * flat[i - 1] ** 2) + 1e-6
    resid = flat / vol
    idx = rng.choice(len(resid), size=n_samples * ref.shape[1], replace=True)
    sampled = resid[idx].reshape(n_samples, ref.shape[1], ref.shape[2])
    vol_idx = rng.choice(len(vol), size=n_samples * ref.shape[1], replace=True)
    sampled_vol = vol[vol_idx].reshape(n_samples, ref.shape[1], ref.shape[2])
    return sampled * sampled_vol

def gaussian_factor_same_stack(n_samples, seed=SEED):
    raw = np.asarray(target_factor_raw, dtype=np.float32)
    flat = raw.reshape(-1, raw.shape[-1])
    rng = np.random.default_rng(seed)
    mu = flat.mean(axis=0)
    cov = np.cov(flat.T)
    diag = np.diag(np.diag(cov))
    cov = 0.95 * cov + 0.05 * diag
    L = np.linalg.cholesky(cov + 1e-6 * np.eye(cov.shape[0]))
    z = rng.normal(size=(n_samples, WINDOW_SIZE, raw.shape[-1])).astype(np.float32)
    factor_raw = z @ L.T + mu
    factor_raw = calibrate_factor_windows(factor_raw, target_factor_raw)
    returns, _ = reconstruct_asset_windows_from_factors(factor_raw, add_residual=True, stress=False, seed=seed + 17)
    return to_numpy(returns)

def metric_bundle(name, samples, real_ref):
    weights = np.ones(to_numpy(samples).shape[-1]) / to_numpy(samples).shape[-1]
    port, daily = portfolio_terminal_returns(samples, weights)
    var5 = np.quantile(port, 0.05)
    var1 = np.quantile(port, 0.01)
    cvar5 = port[port <= var5].mean()
    dd = max_drawdown(daily)
    out = {
        "model": name,
        "distribution_coverage": distribution_coverage(real_ref, samples),
        "mmd_rbf_projected": mmd_rbf_projected(real_ref, samples, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS),
        "terminal_mean": float(np.mean(port)),
        "terminal_median": float(np.median(port)),
        "var5": float(var5),
        "var1": float(var1),
        "cvar5": float(cvar5),
        "probability_loss": float((port < 0).mean()),
        "probability_drawdown_10pct": float((dd <= -0.10).mean()),
    }
    out.update(corr_stats(real_ref, samples))
    return out

weights = np.ones(N_ASSETS_ACTUAL) / N_ASSETS_ACTUAL
train_all_reference = dataset_asset_windows(asset_train_ds, max_samples=min(len(asset_train_ds), BASELINE_SCENARIOS), seed=SEED + 3)
train_target_reference = target_real_returns if len(target_real_returns) >= 64 else train_all_reference

if len(real_valid_target_returns) >= 64:
    real_eval_returns = real_valid_target_returns
    eval_reference_name = "valid_target_regime"
    baseline_train_reference = train_target_reference
else:
    real_eval_returns = real_valid_returns
    eval_reference_name = "valid_all_regimes"
    baseline_train_reference = train_all_reference

real_eval_returns_for_metrics = strided_windows_np(real_eval_returns, EVAL_WINDOW_STRIDE)
CONFIG["eval_window_stride_applied"] = int(EVAL_WINDOW_STRIDE)
CONFIG["eval_windows_are_overlapping"] = bool(EVAL_WINDOW_STRIDE < WINDOW_SIZE)
n_base = min(BASELINE_SCENARIOS, len(real_eval_returns_for_metrics), len(synthetic_returns), len(synthetic_returns_base), len(baseline_train_reference))
if n_base < 64:
    raise RuntimeError(f"Too few evaluation windows for robust metrics: {n_base}")

bootstrap_returns = historical_bootstrap_windows(baseline_train_reference, n_base, seed=SEED + 7)
gaussian_returns = gaussian_cov_from_windows(baseline_train_reference, n_base, WINDOW_SIZE, seed=SEED + 11)
t_copula_returns = t_copula_from_windows(baseline_train_reference, n_base, WINDOW_SIZE, seed=SEED + 13)
fhs_returns = filtered_historical_simulation(baseline_train_reference, n_base, seed=SEED + 19)
same_stack_gaussian_returns = gaussian_factor_same_stack(n_base, seed=SEED + 23)

model_samples = {
    "factor_ddpm_stress": sample_windows_np(synthetic_returns, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED),
    "factor_ddpm_base": sample_windows_np(synthetic_returns_base, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED),
    "historical_bootstrap_train": bootstrap_returns,
    "gaussian_cov_train": gaussian_returns,
    "t_copula_train": t_copula_returns,
    "filtered_historical_simulation_train": fhs_returns,
    "gaussian_factor_same_calibration_stack": same_stack_gaussian_returns,
}
if "synthetic_returns_uncalibrated" in globals():
    model_samples["factor_ddpm_stress_uncalibrated"] = sample_windows_np(synthetic_returns_uncalibrated, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED)
if "synthetic_returns_base_uncalibrated" in globals():
    model_samples["factor_ddpm_base_uncalibrated"] = sample_windows_np(synthetic_returns_base_uncalibrated, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED)
CONFIG["eval_uses_synthetic_subsample"] = True
CONFIG["primary_eval_subsample_seed"] = int(PRIMARY_EVAL_SUBSAMPLE_SEED)

metrics_table = pd.DataFrame(
    [metric_bundle(name, samples, real_eval_returns_for_metrics[:n_base]) for name, samples in model_samples.items()]
).set_index("model")

mmd_stability_rows = []
for name, samples in model_samples.items():
    row = {"model": name}
    row.update(mmd_rbf_projected_multi(real_eval_returns_for_metrics[:n_base], samples, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS))
    mmd_stability_rows.append(row)
mmd_stability_table = pd.DataFrame(mmd_stability_rows).set_index("model")
for col in mmd_stability_table.columns:
    metrics_table[col] = mmd_stability_table[col]

candidate_model = "factor_ddpm_base"
stress_port, stress_daily = portfolio_terminal_returns(synthetic_returns, weights)
base_port, base_daily = portfolio_terminal_returns(synthetic_returns_base, weights)
syn_port, syn_daily = stress_port, stress_daily  # stress diagnostics and walk-forward coverage use the adverse book
real_port, real_daily = portfolio_terminal_returns(real_eval_returns, weights)
real_valid_port_all, real_valid_daily_all = portfolio_terminal_returns(real_valid_returns, weights)

train_target_metrics = pd.Series(quick_portfolio_summary(train_target_reference), name="train_target_empirical")
valid_eval_metrics = pd.Series(quick_portfolio_summary(real_eval_returns), name=eval_reference_name)
synthetic_stress_metrics = pd.Series(quick_portfolio_summary(synthetic_returns), name="factor_ddpm_stress")
synthetic_base_metrics = pd.Series(quick_portfolio_summary(synthetic_returns_base), name="factor_ddpm_base")
candidate_metrics_series = synthetic_base_metrics if candidate_model == "factor_ddpm_base" else synthetic_stress_metrics
regime_calibration = pd.concat([train_target_metrics, valid_eval_metrics, synthetic_stress_metrics, synthetic_base_metrics], axis=1)

metrics = metrics_table.loc[candidate_model].to_dict()
metrics.update({f"train_target_{k}": float(v) for k, v in train_target_metrics.items()})
metrics.update({f"eval_reference_{k}": float(v) for k, v in valid_eval_metrics.items()})
metrics.update({f"synthetic_{k}": float(v) for k, v in candidate_metrics_series.items()})
metrics.update({f"stress_{k}": float(v) for k, v in synthetic_stress_metrics.items()})
metrics["candidate_model"] = candidate_model
metrics["stress_model_for_walk_forward"] = "factor_ddpm_stress"
metrics["selected_guidance_scale"] = float(CONFIG.get("selected_guidance_scale", GUIDANCE_SCALE))
metrics["eval_reference_name"] = eval_reference_name
metrics["n_eval_windows"] = int(n_base)
metrics["n_valid_target_windows_available"] = int(len(real_valid_target_returns))
CONFIG["candidate_model"] = candidate_model
CONFIG["stress_model_for_walk_forward"] = "factor_ddpm_stress"
CONFIG["eval_reference_name"] = eval_reference_name
CONFIG["baselines_use_validation_for_generation"] = False
CONFIG["mmd_standardization"] = "real_reference_only"

def period_portfolio_terminals(returns_frame, start, end, columns, window_size):
    period = returns_frame.loc[pd.Timestamp(start):pd.Timestamp(end), list(columns)].dropna(how="any")
    if len(period) < max(5, window_size // 2):
        return np.array([])
    arr = period.values.astype(np.float32)
    if len(arr) < window_size:
        windows = arr[None, :, :]
    else:
        windows = np.stack([arr[i:i + window_size] for i in range(0, len(arr) - window_size + 1)])
    weights = np.ones(windows.shape[-1]) / windows.shape[-1]
    daily = np.clip(windows @ weights, -0.95, 1.50)
    return np.prod(1 + daily, axis=1) - 1

def walk_forward_crisis_audit(synthetic_port):
    crisis_periods = {
        "covid_crash_2020": ("2020-02-18", "2020-04-30"),
        "inflation_bear_2022": ("2022-01-03", "2022-10-31"),
        "bank_stress_2023": ("2023-03-01", "2023-04-30"),
    }
    rows = []
    syn_q01, syn_q05, syn_q50 = np.quantile(synthetic_port, [0.01, 0.05, 0.50])
    for name, (start, end) in crisis_periods.items():
        actual = period_portfolio_terminals(returns_df, start, end, train_ds.columns, WINDOW_SIZE)
        if len(actual) == 0:
            continue
        rows.append({
            "period": name,
            "start": start,
            "end": end,
            "n_actual_windows": int(len(actual)),
            "actual_min_terminal": float(np.min(actual)),
            "actual_median_terminal": float(np.median(actual)),
            "actual_mean_terminal": float(np.mean(actual)),
            "synthetic_q01": float(syn_q01),
            "synthetic_q05": float(syn_q05),
            "synthetic_median": float(syn_q50),
            "crisis_min_covered_by_syn_1pct": bool(syn_q01 <= np.min(actual)),
            "crisis_min_covered_by_syn_5pct": bool(syn_q05 <= np.min(actual)),
        })
    return pd.DataFrame(rows)

walk_forward_df = walk_forward_crisis_audit(stress_port)
metrics["walk_forward_periods"] = int(len(walk_forward_df))
metrics["walk_forward_1pct_coverage_rate"] = float(walk_forward_df["crisis_min_covered_by_syn_1pct"].mean()) if len(walk_forward_df) else np.nan
metrics["walk_forward_5pct_coverage_rate"] = float(walk_forward_df["crisis_min_covered_by_syn_5pct"].mean()) if len(walk_forward_df) else np.nan

gaussian_row = metrics_table.loc["gaussian_cov_train"]
candidate_row = metrics_table.loc[candidate_model]
relative_to_gaussian = pd.Series({
    "mmd_ratio_candidate_vs_gaussian": float(candidate_row["mmd_rbf_projected"] / max(gaussian_row["mmd_rbf_projected"], 1e-12)),
    "mmd_multi_ratio_candidate_vs_gaussian": float(candidate_row["mmd_rbf_projected_multi_mean"] / max(gaussian_row["mmd_rbf_projected_multi_mean"], 1e-12)),
    "corr_mae_delta_candidate_minus_gaussian": float(candidate_row["corr_mae"] - gaussian_row["corr_mae"]),
    "eigen_rmse_delta_candidate_minus_gaussian": float(candidate_row["corr_top20_eigen_rmse"] - gaussian_row["corr_top20_eigen_rmse"]),
    "candidate_corr_near_gaussian_within_tol": bool(candidate_row["corr_mae"] <= gaussian_row["corr_mae"] + CORR_MAE_NEAR_GAUSSIAN_TOL),
    "candidate_mmd_ratio_within_research_gate": bool(candidate_row["mmd_rbf_projected_multi_mean"] <= gaussian_row["mmd_rbf_projected_multi_mean"] * MMD_RATIO_MAX_RESEARCH),
})
metrics.update({k: (bool(v) if isinstance(v, (bool, np.bool_)) else float(v)) for k, v in relative_to_gaussian.items()})

calibration_audit_table = metrics_table.loc[
    [idx for idx in ["factor_ddpm_base_uncalibrated", "factor_ddpm_base", "factor_ddpm_stress_uncalibrated", "factor_ddpm_stress"] if idx in metrics_table.index],
    ["mmd_rbf_projected", "mmd_rbf_projected_multi_mean", "corr_mae", "corr_top20_eigen_rmse", "cvar5", "probability_drawdown_10pct"],
].copy()

subsample_rows = []
for seed in parse_int_list(EVAL_SYNTHETIC_SUBSAMPLE_SEEDS):
    base_subset = sample_windows_np(synthetic_returns_base, n_base, seed)
    stress_subset = sample_windows_np(synthetic_returns, n_base, seed)
    row = {
        "subsample_seed": int(seed),
        "base_mmd": mmd_rbf_projected(real_eval_returns_for_metrics[:n_base], base_subset, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS, seed=seed),
        "base_mmd_multi_mean": mmd_rbf_projected_multi(real_eval_returns_for_metrics[:n_base], base_subset, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS)["mmd_rbf_projected_multi_mean"],
        "base_corr_mae": corr_stats(real_eval_returns_for_metrics[:n_base], base_subset)["corr_mae"],
        "stress_mmd": mmd_rbf_projected(real_eval_returns_for_metrics[:n_base], stress_subset, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS, seed=seed),
        "stress_q01": float(np.quantile(portfolio_terminal_returns(stress_subset, weights)[0], 0.01)),
    }
    subsample_rows.append(row)
synthetic_subsample_sensitivity = pd.DataFrame(subsample_rows)
metrics["subsample_base_mmd_multi_mean_mean"] = float(synthetic_subsample_sensitivity["base_mmd_multi_mean"].mean())
metrics["subsample_base_mmd_multi_mean_std"] = float(synthetic_subsample_sensitivity["base_mmd_multi_mean"].std())
metrics["subsample_base_corr_mae_mean"] = float(synthetic_subsample_sensitivity["base_corr_mae"].mean())
CONFIG["eval_synthetic_subsample_seeds"] = EVAL_SYNTHETIC_SUBSAMPLE_SEEDS

real_eval_factor_norm = dataset_factor_windows(
    valid_ds,
    max_samples=min(len(valid_ds), n_base),
    regime_filter=TARGET_REGIME if eval_reference_name == "valid_target_regime" else None,
)
if len(real_eval_factor_norm) < 64:
    real_eval_factor_norm = dataset_factor_windows(valid_ds, max_samples=min(len(valid_ds), n_base))
real_eval_factor_raw = denormalize_factor_windows(real_eval_factor_norm)
factor_n = min(len(real_eval_factor_raw), len(synthetic_factor_raw), n_base)
factor_space_metrics = pd.Series({
    "factor_mmd_base": mmd_rbf_projected(real_eval_factor_raw[:factor_n], synthetic_factor_raw[:factor_n], proj_dim=min(MMD_PROJECTION_DIM, N_FACTORS_ACTUAL), max_n=factor_n),
    "factor_mmd_base_multi_mean": mmd_rbf_projected_multi(real_eval_factor_raw[:factor_n], synthetic_factor_raw[:factor_n], proj_dim=min(MMD_PROJECTION_DIM, N_FACTORS_ACTUAL), max_n=factor_n)["mmd_rbf_projected_multi_mean"],
    "factor_eval_windows": int(factor_n),
}, name="value")
metrics.update({k: float(v) for k, v in factor_space_metrics.items()})

stress_multiplier_counts = {str(float(k)): int(v) for k, v in zip(*np.unique(stress_multipliers, return_counts=True))}
stress_q = float(np.quantile(stress_port, ENDPOINT_STRESS_QUANTILE))
endpoint_contract = {
    "endpoint": "/api/v1/workspaces/{workspaceId}/market-simulation",
    "status": "research_champion_offline_only",
    "base_model": "factor_ddpm_base",
    "stress_model": "factor_ddpm_stress",
    "generated_scenarios": int(len(stress_port)),
    "endpoint_default_scenarios": int(ENDPOINT_DEFAULT_SCENARIOS),
    "endpoint_min_scenarios": int(ENDPOINT_MIN_SCENARIOS),
    "endpoint_min_stress_scenarios": int(ENDPOINT_MIN_STRESS_SCENARIOS),
    "endpoint_stress_quantile": float(ENDPOINT_STRESS_QUANTILE),
    "endpoint_small_request_policy": ENDPOINT_SMALL_REQUEST_POLICY,
    "stress_stratified_sampling": bool(STRESS_STRATIFIED_SAMPLING),
    "stress_multiplier_counts": json.dumps(stress_multiplier_counts, sort_keys=True),
    "stress_book_q01": stress_q,
    "scenario_count_ok_for_stress_endpoint": bool(len(stress_port) >= ENDPOINT_MIN_STRESS_SCENARIOS),
    "ready_for_endpoint_requires": "strict MMD and correlation gates; current research champion is offline only",
}
endpoint_contract_table = pd.Series(endpoint_contract, name="value").to_frame()
metrics["endpoint_scenario_count_ok"] = bool(endpoint_contract["scenario_count_ok_for_stress_endpoint"])
metrics["endpoint_stress_book_q01"] = float(stress_q)
CONFIG["endpoint_contract_status"] = endpoint_contract["status"]
CONFIG["endpoint_contract"] = endpoint_contract

print("Evaluation reference:", eval_reference_name, "| windows:", n_base, "| candidate:", candidate_model)
display(metrics_table)

def make_portfolio_weight_suite(n_assets, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = [("equal_weight", np.ones(n_assets) / n_assets)]
    for i in range(PORTFOLIO_TEST_COUNT):
        rows.append((f"random_dirichlet_{i+1:02d}", rng.dirichlet(np.ones(n_assets))))
    for level in parse_float_list(PORTFOLIO_CONCENTRATION_LEVELS):
        w = np.ones(n_assets) * ((1 - level) / max(n_assets - 1, 1))
        w[int(rng.integers(0, n_assets))] = level
        rows.append((f"single_name_{float(level):.0%}", w / w.sum()))
    return rows

portfolio_suite_rows = []
for portfolio_name, portfolio_weights in make_portfolio_weight_suite(N_ASSETS_ACTUAL):
    for model_name, samples in model_samples.items():
        port, daily = portfolio_terminal_returns(samples, portfolio_weights)
        var5_suite = np.quantile(port, 0.05)
        var1_suite = np.quantile(port, 0.01)
        portfolio_suite_rows.append({
            "portfolio": portfolio_name,
            "model": model_name,
            "var5": float(var5_suite),
            "var1": float(var1_suite),
            "cvar5": float(port[port <= var5_suite].mean()),
            "probability_drawdown_10pct": float((max_drawdown(daily) <= -0.10).mean()),
        })
portfolio_suite_table = pd.DataFrame(portfolio_suite_rows)
display(portfolio_suite_table.head(20))
display(metrics_table.T)
display(regime_calibration)
display(walk_forward_df)
display(relative_to_gaussian.to_frame("value"))
display(calibration_audit_table)
display(synthetic_subsample_sensitivity)
display(factor_space_metrics.to_frame())
display(endpoint_contract_table)


In [ ]:
#@title Tail contributors, diagnostic plots, and endpoint gate
var5 = np.quantile(syn_port, 0.05)
cvar5 = syn_port[syn_port <= var5].mean()
tail_mask = syn_port <= var5
asset_terminal = np.prod(1 + to_numpy(synthetic_returns), axis=1) - 1
contrib = pd.Series(asset_terminal[tail_mask].mean(axis=0) * weights, index=train_ds.columns).sort_values()
display(contrib.head(20).to_frame("weighted_tail_contribution"))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

axes[0, 0].hist(real_port, bins=50, alpha=0.45, label=eval_reference_name)
axes[0, 0].hist(syn_port, bins=50, alpha=0.45, label="factor-ddpm stress")
axes[0, 0].hist(base_port, bins=50, alpha=0.30, label="factor-ddpm base")
axes[0, 0].axvline(var5, color="red", linestyle="--", label="Stress VaR 5%")
axes[0, 0].axvline(cvar5, color="black", linestyle=":", label="Stress CVaR 5%")
axes[0, 0].set_title("Terminal equal-weight portfolio returns")
axes[0, 0].legend(fontsize=8)

real_corr = np.nan_to_num(np.corrcoef(to_numpy(real_eval_returns).reshape(-1, N_ASSETS_ACTUAL).T), nan=0.0)
syn_corr = np.nan_to_num(np.corrcoef(to_numpy(synthetic_returns).reshape(-1, N_ASSETS_ACTUAL).T), nan=0.0)
im = axes[0, 1].imshow(syn_corr - real_corr, vmin=-0.5, vmax=0.5, cmap="coolwarm")
axes[0, 1].set_title("Stress synthetic minus eval-reference correlation")
plt.colorbar(im, ax=axes[0, 1], shrink=0.75)

q = np.linspace(0.01, 0.99, 99)
axes[0, 2].plot(np.quantile(real_port, q), np.quantile(syn_port, q), marker=".", linestyle="none", label="stress")
axes[0, 2].plot(np.quantile(real_port, q), np.quantile(base_port, q), marker=".", linestyle="none", alpha=0.45, label="base")
lims = [min(np.quantile(real_port, 0.01), np.quantile(syn_port, 0.01), np.quantile(base_port, 0.01)), max(np.quantile(real_port, 0.99), np.quantile(syn_port, 0.99), np.quantile(base_port, 0.99))]
axes[0, 2].plot(lims, lims, color="black", linewidth=1)
axes[0, 2].set_title("QQ: eval reference vs factor DDPM")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.25)

for i in range(min(25, len(syn_daily))):
    axes[1, 0].plot(np.cumprod(1 + syn_daily[i]) - 1, alpha=0.22)
axes[1, 0].set_title("Stress sample portfolio paths")
axes[1, 0].grid(True, alpha=0.25)

axes[1, 1].hist(stress_multipliers, bins=np.unique(stress_multipliers).size)
axes[1, 1].set_title("Stress multiplier mixture")
axes[1, 1].grid(True, alpha=0.25)

eig_real = np.sort(np.linalg.eigvalsh(real_corr))[-20:]
eig_syn = np.sort(np.linalg.eigvalsh(syn_corr))[-20:]
axes[1, 2].plot(eig_real, label="real top eigen")
axes[1, 2].plot(eig_syn, label="synthetic top eigen")
axes[1, 2].set_title("Correlation eigenstructure")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

cvar_ref_col = eval_reference_name
hist_for_gate = pd.DataFrame(history)
total_batches_seen = max(int(len(hist_for_gate) * math.ceil(len(train_loader))), 1)
total_skipped_batches = int(hist_for_gate["skipped_batches"].sum()) if "skipped_batches" in hist_for_gate else 0
skipped_batch_rate = total_skipped_batches / total_batches_seen

scorecard = {
    "beats_gaussian_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected"] < metrics_table.loc["gaussian_cov_train", "mmd_rbf_projected"]),
    "beats_gaussian_mmd_multi": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["gaussian_cov_train", "mmd_rbf_projected_multi_mean"]),
    "mmd_ratio_within_research_gate": bool(metrics.get("mmd_multi_ratio_candidate_vs_gaussian", np.inf) <= MMD_RATIO_MAX_RESEARCH),
    "beats_gaussian_corr": bool(metrics_table.loc[candidate_model, "corr_mae"] < metrics_table.loc["gaussian_cov_train", "corr_mae"]),
    "beats_t_copula_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["t_copula_train", "mmd_rbf_projected_multi_mean"]),
    "beats_fhs_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["filtered_historical_simulation_train", "mmd_rbf_projected_multi_mean"]),
    "beats_same_stack_gaussian_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["gaussian_factor_same_calibration_stack", "mmd_rbf_projected_multi_mean"]),
    "corr_near_gaussian": bool(metrics_table.loc[candidate_model, "corr_mae"] <= metrics_table.loc["gaussian_cov_train", "corr_mae"] + CORR_MAE_NEAR_GAUSSIAN_TOL),
    "corr_fidelity_ge_0_80": bool(metrics_table.loc[candidate_model, "correlation_fidelity"] >= 0.80),
    "target_cvar_close_to_eval_reference": bool(abs(regime_calibration.loc["cvar5", candidate_model] - regime_calibration.loc["cvar5", cvar_ref_col]) <= max(0.01, abs(regime_calibration.loc["cvar5", cvar_ref_col]) * 0.35)),
    "stress_walk_forward_1pct_covers_all": bool(len(walk_forward_df) > 0 and walk_forward_df["crisis_min_covered_by_syn_1pct"].all()),
    "walk_forward_1pct_covers_all": bool(len(walk_forward_df) > 0 and walk_forward_df["crisis_min_covered_by_syn_1pct"].all()),
    "stress_stratified_sampling": bool(CONFIG.get("stress_stratified_sampling_applied", False)),
    "endpoint_scenario_count_ok": bool(metrics.get("endpoint_scenario_count_ok", False)),
    "skipped_batch_rate_ok": bool(skipped_batch_rate <= SKIPPED_BATCH_RATE_MAX),
    "no_validation_used_for_guidance_or_cholesky": bool(not CONFIG.get("guidance_uses_validation", True) and not CONFIG.get("cholesky_uses_validation", True)),
    "no_full_window_stress_floor": bool(CONFIG.get("stress_full_window_floor_applied") is False),
    "non_overlapping_eval_windows": bool(CONFIG.get("eval_window_stride_applied", 1) >= WINDOW_SIZE),
}
scorecard["research_champion"] = bool(
    scorecard["mmd_ratio_within_research_gate"]
    and scorecard["beats_gaussian_mmd_multi"]
    and scorecard["beats_t_copula_mmd"]
    and scorecard["beats_fhs_mmd"]
    and scorecard["beats_same_stack_gaussian_mmd"]
    and scorecard["corr_near_gaussian"]
    and scorecard["corr_fidelity_ge_0_80"]
    and scorecard["no_full_window_stress_floor"]
    and scorecard["non_overlapping_eval_windows"]
    and scorecard["target_cvar_close_to_eval_reference"]
    and scorecard["walk_forward_1pct_covers_all"]
    and scorecard["stress_stratified_sampling"]
    and scorecard["endpoint_scenario_count_ok"]
    and scorecard["skipped_batch_rate_ok"]
    and scorecard["no_validation_used_for_guidance_or_cholesky"]
)
scorecard["ready_for_endpoint"] = bool(
    scorecard["beats_gaussian_mmd_multi"]
    and scorecard["beats_gaussian_corr"]
    and scorecard["corr_fidelity_ge_0_80"]
    and scorecard["target_cvar_close_to_eval_reference"]
    and scorecard["walk_forward_1pct_covers_all"]
    and scorecard["skipped_batch_rate_ok"]
    and scorecard["no_validation_used_for_guidance_or_cholesky"]
)
metrics["skipped_batch_rate"] = float(skipped_batch_rate)
metrics["total_skipped_batches"] = int(total_skipped_batches)
metrics["total_batches_seen"] = int(total_batches_seen)
metrics["research_champion"] = bool(scorecard["research_champion"])
metrics["ready_for_endpoint"] = bool(scorecard["ready_for_endpoint"])
metrics["scorecard"] = scorecard
CONFIG["research_champion"] = bool(scorecard["research_champion"])
CONFIG["ready_for_endpoint"] = bool(scorecard["ready_for_endpoint"])
display(pd.Series(scorecard).to_frame("pass"))
print("Skipped batch rate:", skipped_batch_rate, "| skipped:", total_skipped_batches, "/", total_batches_seen)


## 9. Save Artifacts to Drive

The checkpoint is already saved during training. This cell writes:
- config JSON;
- metrics JSON;
- synthetic scenarios as compressed NumPy;
- tail contribution CSV;
- training history CSV.
        


In [ ]:
#@title Save artifacts
run_id = time.strftime("%Y%m%d_%H%M%S")
run_dir = ARTIFACT_DIR / f"factor_ddpm_run_{run_id}"
run_dir.mkdir(parents=True, exist_ok=True)

with open(run_dir / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

scenario_payload = dict(
    synthetic_returns=to_numpy(synthetic_returns),
    synthetic_returns_base=to_numpy(synthetic_returns_base),
    synthetic_factor_raw=synthetic_factor_raw,
    synthetic_factor_norm=to_numpy(synthetic_factor_norm),
    stress_multipliers=stress_multipliers,
    columns=np.array(train_ds.columns),
    factor_columns=np.array(FACTOR_COLUMNS),
    target_regime=np.array([TARGET_REGIME]),
    selected_guidance_scale=np.array([CONFIG.get("selected_guidance_scale", GUIDANCE_SCALE)]),
    eval_reference_name=np.array([CONFIG.get("eval_reference_name", "unknown")]),
    ready_for_endpoint=np.array([bool(CONFIG.get("ready_for_endpoint", False))]),
)
if "synthetic_factor_raw_uncalibrated" in globals():
    scenario_payload["synthetic_factor_raw_uncalibrated"] = synthetic_factor_raw_uncalibrated
if "synthetic_returns_base_uncalibrated" in globals():
    scenario_payload["synthetic_returns_base_uncalibrated"] = to_numpy(synthetic_returns_base_uncalibrated)
if "synthetic_returns_uncalibrated" in globals():
    scenario_payload["synthetic_returns_uncalibrated"] = to_numpy(synthetic_returns_uncalibrated)
np.savez_compressed(run_dir / "synthetic_scenarios.npz", **scenario_payload)

factor_bank_path = None
if EXPORT_FACTOR_SCENARIO_BANK:
    bank_dtype = np.float16 if FACTOR_BANK_DTYPE == "float16" else np.float32
    factor_bank_payload = {
        "factor_paths_base": np.asarray(synthetic_factor_raw, dtype=bank_dtype),
        "factor_paths_stress": np.asarray(synthetic_factor_raw, dtype=bank_dtype),
        "stress_multipliers": np.asarray(stress_multipliers, dtype=np.float32),
        "factor_columns": np.array(FACTOR_COLUMNS),
        "window_size": np.array([WINDOW_SIZE]),
        "target_regime": np.array([TARGET_REGIME]),
        "run_id": np.array([run_id]),
        "shock_mode": np.array([STRESS_SHOCK_MODE]),
    }
    factor_bank_path = run_dir / "factor_scenario_bank_fp16.npz"
    np.savez_compressed(factor_bank_path, **factor_bank_payload)
pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)
metrics_table.to_csv(run_dir / "metrics_table.csv")
regime_calibration.to_csv(run_dir / "regime_calibration.csv")
if "walk_forward_df" in globals() and len(walk_forward_df):
    walk_forward_df.to_csv(run_dir / "walk_forward_crisis_audit.csv", index=False)
if "guidance_sweep_df" in globals() and len(guidance_sweep_df):
    guidance_sweep_df.to_csv(run_dir / "guidance_sweep.csv", index=False)
if "mmd_stability_table" in globals():
    mmd_stability_table.to_csv(run_dir / "mmd_stability_table.csv")
if "calibration_audit_table" in globals():
    calibration_audit_table.to_csv(run_dir / "calibration_audit_table.csv")
if "synthetic_subsample_sensitivity" in globals():
    synthetic_subsample_sensitivity.to_csv(run_dir / "synthetic_subsample_sensitivity.csv", index=False)
if "portfolio_suite_table" in globals():
    portfolio_suite_table.to_csv(run_dir / "portfolio_suite_table.csv", index=False)
if "endpoint_contract_table" in globals():
    endpoint_contract_table.to_csv(run_dir / "endpoint_contract.csv")
    with open(run_dir / "endpoint_contract.json", "w") as f:
        json.dump(endpoint_contract, f, indent=2)
if "factor_space_metrics" in globals():
    factor_space_metrics.to_frame("value").to_csv(run_dir / "factor_space_metrics.csv")
contrib.to_csv(run_dir / "tail_contributors.csv")

factor_package = {
    "factor_columns": FACTOR_COLUMNS,
    "sector_names": sector_names,
    "pca_explained_variance_ratio_sum": CONFIG.get("pca_explained_variance_ratio_sum"),
    "recon_beta_shape": list(RECON_BETA.shape),
    "residual_bootstrap_scale": RESIDUAL_BOOTSTRAP_SCALE,
    "stress_mix_weights": STRESS_MIX_WEIGHTS,
    "stress_mix_multipliers": STRESS_MIX_MULTIPLIERS,
    "factor_calibration_applied": bool(CONFIG.get("factor_calibration_applied", False)),
    "factor_calibration_alpha": float(CONFIG.get("factor_calibration_alpha", 0.0)),
    "factor_calibration_shrinkage": float(CONFIG.get("factor_calibration_shrinkage", 0.0)),
    "factor_calibration_target_source": CONFIG.get("factor_calibration_target_source", "unknown"),
    "cholesky_calibration_applied": bool(CONFIG.get("cholesky_calibration_applied", False)),
    "cholesky_calibration_alpha": float(CONFIG.get("cholesky_calibration_alpha", 0.0)),
    "cholesky_shrinkage": float(CONFIG.get("cholesky_shrinkage", 0.0)),
    "cholesky_calibration_target_source": CONFIG.get("cholesky_calibration_target_source", "unknown"),
    "stress_shock_mode": CONFIG.get("stress_shock_mode_applied", STRESS_SHOCK_MODE),
    "stress_full_window_floor_applied": bool(CONFIG.get("stress_full_window_floor_applied", False)),
    "survivorship_disclosure": SURVIVORSHIP_DISCLOSURE,
}
with open(run_dir / "factor_package.json", "w") as f:
    json.dump(factor_package, f, indent=2)

api_manifest = {
    "endpoint": "/api/v1/workspaces/{workspaceId}/market-simulation",
    "model_family": MODEL_FAMILY,
    "candidate_model": CONFIG.get("candidate_model", "factor_ddpm_base"),
    "research_champion": bool(CONFIG.get("research_champion", False)),
    "ready_for_endpoint": bool(CONFIG.get("ready_for_endpoint", False)),
    "endpoint_gate": "deploy only when ready_for_endpoint is true; research_champion permits offline use only",
    "checkpoint_path": str(CHECKPOINT_DIR / "best_factor_ddpm_market_simulator.pt"),
    "artifact_dir": str(run_dir),
    "columns_path": str(run_dir / "synthetic_scenarios.npz"),
    "config_path": str(run_dir / "config.json"),
    "metrics_path": str(run_dir / "metrics.json"),
    "factor_package_path": str(run_dir / "factor_package.json"),
    "endpoint_contract_path": str(run_dir / "endpoint_contract.json"),
    "factor_scenario_bank_path": str(factor_bank_path) if factor_bank_path else None,
    "endpoint_serving_strategy": "static_factor_bank_projection",
    "selected_guidance_scale": float(CONFIG.get("selected_guidance_scale", GUIDANCE_SCALE)),
    "target_regime": int(TARGET_REGIME),
    "n_assets_actual": int(N_ASSETS_ACTUAL),
    "n_factors_actual": int(N_FACTORS_ACTUAL),
    "n_macro_features": int(N_MACRO_FEATURES),
    "eval_reference_name": CONFIG.get("eval_reference_name", "unknown"),
    "validation_protocol": CONFIG.get("validation_protocol", "unknown"),
    "guidance_uses_validation": bool(CONFIG.get("guidance_uses_validation", True)),
    "cholesky_uses_validation": bool(CONFIG.get("cholesky_uses_validation", True)),
    "stress_scenario_mode_applied": bool(CONFIG.get("stress_scenario_mode_applied", False)),
    "stress_stratified_sampling_applied": bool(CONFIG.get("stress_stratified_sampling_applied", False)),
    "stress_multiplier_counts": CONFIG.get("stress_multiplier_counts", {}),
    "stress_model_for_walk_forward": CONFIG.get("stress_model_for_walk_forward", "factor_ddpm_stress"),
    "skipped_batch_rate": float(metrics.get("skipped_batch_rate", np.nan)),
    "mmd_multi_ratio_candidate_vs_gaussian": float(metrics.get("mmd_multi_ratio_candidate_vs_gaussian", np.nan)),
    "eval_uses_synthetic_subsample": bool(CONFIG.get("eval_uses_synthetic_subsample", False)),
    "primary_eval_subsample_seed": int(CONFIG.get("primary_eval_subsample_seed", -1)),
    "subsample_base_mmd_multi_mean_mean": float(metrics.get("subsample_base_mmd_multi_mean_mean", np.nan)),
    "subsample_base_mmd_multi_mean_std": float(metrics.get("subsample_base_mmd_multi_mean_std", np.nan)),
    "endpoint_contract_status": CONFIG.get("endpoint_contract_status", "unknown"),
    "endpoint_default_scenarios": int(CONFIG.get("endpoint_default_scenarios", -1)),
    "endpoint_min_scenarios": int(CONFIG.get("endpoint_min_scenarios", -1)),
    "endpoint_min_stress_scenarios": int(CONFIG.get("endpoint_min_stress_scenarios", -1)),
    "endpoint_stress_book_q01": float(metrics.get("endpoint_stress_book_q01", np.nan)),
    "factor_calibration_applied": bool(CONFIG.get("factor_calibration_applied", False)),
    "factor_calibration_uses_validation": bool(CONFIG.get("factor_calibration_uses_validation", True)),
    "factor_calibration_target_source": CONFIG.get("factor_calibration_target_source", "unknown"),
    "cholesky_calibration_applied": bool(CONFIG.get("cholesky_calibration_applied", False)),
    "cholesky_calibration_target_source": CONFIG.get("cholesky_calibration_target_source", "unknown"),
}
with open(run_dir / "blsprime_market_simulation_manifest.json", "w") as f:
    json.dump(api_manifest, f, indent=2)

print("Saved artifacts to:", run_dir)
print("Best checkpoint:", CHECKPOINT_DIR / "best_factor_ddpm_market_simulator.pt")
print("API manifest:", run_dir / "blsprime_market_simulation_manifest.json")
print("Research champion:", api_manifest["research_champion"])
print("Ready for endpoint:", api_manifest["ready_for_endpoint"])


## 10. Promotion Rule

Promote only if `ready_for_endpoint == True`. This Factor-DDPM notebook is allowed to generate both base and stress scenarios, but the endpoint manifest stays gated until it beats Gaussian covariance on MMD/correlation, matches target tail risk, covers walk-forward crises, and has zero skipped batches.


## V8 interpretation rule

If the same-stack Gaussian, t-copula, FHS, or bootstrap baselines beat the DDPM on the gated metrics, do not promote the DDPM. The honest product in that case is a calibrated factor stress engine, and the diffusion model remains an offline research candidate.
